## Data Clean for PWC Central Data Hub
*Date: Mar 16th, 2026*

In [1]:
import pandas as pd
import numpy as np
import re

### Demographics
#### Total Population

In [8]:
population_2024 = pd.read_csv('Demographics/Total Population/ACSDT5Y2024.csv')
population_2024.loc[0, "Label (Grouping)"] = 2024
population_2023 = pd.read_csv('Demographics/Total Population/ACSDT5Y2023.csv')
population_2023.loc[0, "Label (Grouping)"] = 2023
population_2022 = pd.read_csv('Demographics/Total Population/ACSDT5Y2022.csv')
population_2022.loc[0, "Label (Grouping)"] = 2022   
population_2021 = pd.read_csv('Demographics/Total Population/ACSDT5Y2021.csv')
population_2021.loc[0, "Label (Grouping)"] = 2021
population_2020 = pd.read_csv('Demographics/Total Population/ACSDT5Y2020.csv')
population_2020.loc[0, "Label (Grouping)"] = 2020

In [37]:
datasets = [population_2024, population_2023, population_2022, population_2021, population_2020]
population = pd.DataFrame([df.values.flatten() for df in datasets])
population.columns = population_2024.columns
population['Variable'] = 'Total Population'
population = population.rename(columns={'Label (Grouping)': 'Year'})
cols_to_keep = [col for col in population.columns if '!!Estimate' in col and 'Margin of Error' not in col]
population_filtered = population[cols_to_keep].copy()
def clean_header(col):
    match = re.search(r"Census Tract ([\d.]+)", col)
    return match.group(1) if match else col
years = [2024, 2023, 2022, 2021, 2020]
population_filtered.columns = [clean_header(c) for c in population_filtered.columns]
population_long = population_filtered.T
population_long.columns = [f'population_{year}' for year in years]
population_long.index.name = 'censustract'
population_long = population_long.reset_index()
population_long

,censustract,population_2024,population_2023,population_2022,population_2021,population_2020
0,9001,"3,196","3,001","3,234","3,467","3,616"
1,9002.01,"3,117","3,180","2,441","2,338","2,192"
2,9002.02,"3,668","3,520","4,124","4,271","4,254"
3,9002.03,"5,442","4,891","4,589","4,579","4,463"
4,9003.01,"3,750","3,514","3,494","3,442","3,410"
...,...,...,...,...,...,...
88,9017.02,"4,831","4,815","4,641","4,455","4,356"
89,9017.03,"2,750","2,574","2,418","2,770","2,518"
90,9017.04,"7,269","7,116","6,582","7,151","6,971"
91,9019,"7,256","8,141","7,729","7,155","6,702"


In [38]:
population_long.to_csv('Demographics/Total Population/population_cleaned.csv', index=False)

#### Under 5 Years Old

In [31]:
under5_2024 = pd.read_csv('Demographics/Under 5 Years Old/ACSDT5Y2024.B01001.csv')
under5_2024.loc[0, "Label (Grouping)"] = 2024
under5_2023 = pd.read_csv('Demographics/Under 5 Years Old/ACSDT5Y2023.B01001.csv')
under5_2023.loc[0, "Label (Grouping)"] = 2023
under5_2022 = pd.read_csv('Demographics/Under 5 Years Old/ACSDT5Y2022.B01001.csv')
under5_2022.loc[0, "Label (Grouping)"] = 2022   
under5_2021 = pd.read_csv('Demographics/Under 5 Years Old/ACSDT5Y2021.B01001.csv')
under5_2021.loc[0, "Label (Grouping)"] = 2021
under5_2020 = pd.read_csv('Demographics/Under 5 Years Old/ACSDT5Y2020.B01001.csv')
under5_2020.loc[0, "Label (Grouping)"] = 2020

In [50]:
def clean_header(col):
    match = re.search(r"Census Tract ([\d.]+)", col)
    return match.group(1) if match else col

def process_under5(df):
    cols_to_keep = [col for col in df.columns if '!!Estimate' in col and 'Margin of Error' not in col]
    filtered = df.iloc[[0, 2, 26]][cols_to_keep].copy()
    no_commas = filtered.replace(',', '', regex=True)
    numeric = no_commas.apply(pd.to_numeric, errors='coerce')
    numeric.columns = [clean_header(c) for c in numeric.columns]
    cleaned = numeric.T
    cleaned.index.name = 'censustract'
    cleaned = cleaned.reset_index()
    cleaned = cleaned.rename(columns={0: 'total', 2: 'male_under_5', 26: 'female_under_5'})
    return cleaned

under5_dfs = {2024: under5_2024, 2023: under5_2023, 2022: under5_2022, 2021: under5_2021, 2020: under5_2020}

under5_cleaned_list = []
for year, df in under5_dfs.items():
    cleaned = process_under5(df)
    cleaned = cleaned.rename(columns={
        'total': f'total_{year}',
        'male_under_5': f'male_under_5_{year}',
        'female_under_5': f'female_under_5_{year}'
    })
    under5_cleaned_list.append(cleaned)

under5_long = under5_cleaned_list[0]
for df in under5_cleaned_list[1:]:
    under5_long = under5_long.merge(df, on='censustract', how='outer')

for year in [2024, 2023, 2022, 2021, 2020]:
    total_col = f'total_{year}'
    male_col = f'male_under_5_{year}'
    female_col = f'female_under_5_{year}'
    under5_long[f'Under5_Proportion_{year}'] = round(
        (under5_long[male_col] + under5_long[female_col]) / under5_long[total_col].replace(0, np.nan) * 100, 2
    )

under5_long

,censustract,total_2024,male_under_5_2024,female_under_5_2024,total_2023,male_under_5_2023,female_under_5_2023,total_2022,male_under_5_2022,female_under_5_2022,...,male_under_5_2021,female_under_5_2021,total_2020,male_under_5_2020,female_under_5_2020,Under5_Proportion_2024,Under5_Proportion_2023,Under5_Proportion_2022,Under5_Proportion_2021,Under5_Proportion_2020
0,9001,3196,47,103,3001,59,82,3234,92,93,...,189,82,3616,153,53,4.69,4.70,5.72,7.82,5.70
1,9002.01,3117,50,87,3180,39,86,2441,46,33,...,73,58,2192,40,25,4.40,3.93,3.24,5.60,2.97
2,9002.02,3668,83,112,3520,70,98,4124,38,131,...,110,152,4254,142,192,5.32,4.77,4.10,6.13,7.85
3,9002.03,5442,156,355,4891,175,287,4589,171,266,...,94,256,4463,46,245,9.39,9.45,9.52,7.64,6.52
4,9003.01,3750,48,134,3514,58,77,3494,52,117,...,59,93,3410,68,73,4.85,3.84,4.84,4.42,4.13
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
88,9017.02,4831,141,139,4815,128,136,4641,116,138,...,155,114,4356,151,82,5.80,5.48,5.47,6.04,5.35
89,9017.03,2750,31,73,2574,28,68,2418,39,54,...,0,126,2518,0,97,3.78,3.73,3.85,4.55,3.85
90,9017.04,7269,365,368,7116,414,292,6582,505,217,...,450,273,6971,451,356,10.08,9.92,10.97,10.11,11.58
91,9019,7256,506,98,8141,433,529,7729,300,435,...,279,454,6702,305,374,8.32,11.82,9.51,10.24,10.13


In [51]:
under5_long.to_csv('Demographics/Under 5 Years Old/under5_cleaned.csv', index=False)

#### Over 64 Years Old

In [61]:
over64_2024 = pd.read_csv('Demographics/Over 64 Years Old/ACSDT5Y2024.B01001.csv')
over64_2024.loc[0, "Label (Grouping)"] = 2024
over64_2023 = pd.read_csv('Demographics/Over 64 Years Old/ACSDT5Y2023.B01001.csv')
over64_2023.loc[0, "Label (Grouping)"] = 2023
over64_2022 = pd.read_csv('Demographics/Over 64 Years Old/ACSDT5Y2022.B01001.csv')
over64_2022.loc[0, "Label (Grouping)"] = 2022   
over64_2021 = pd.read_csv('Demographics/Over 64 Years Old/ACSDT5Y2021.B01001.csv')
over64_2021.loc[0, "Label (Grouping)"] = 2021
over64_2020 = pd.read_csv('Demographics/Over 64 Years Old/ACSDT5Y2020.B01001.csv')
over64_2020.loc[0, "Label (Grouping)"] = 2020

In [63]:
def clean_header(col):
    match = re.search(r"Census Tract ([\d.]+)", col)
    return match.group(1) if match else col

def process_over64(df):
    cols_to_keep = [col for col in df.columns if '!!Estimate' in col and 'Margin of Error' not in col]
    filtered = df.iloc[[0, 19,20,21,22,23,24, 43,44,45,46,47,48]][cols_to_keep].copy()
    no_commas = filtered.replace(',', '', regex=True)
    numeric = no_commas.apply(pd.to_numeric, errors='coerce')
    numeric.columns = [clean_header(c) for c in numeric.columns]
    cleaned = numeric.T
    cleaned.index.name = 'censustract'
    cleaned = cleaned.reset_index()
    cleaned['male_over_64'] = cleaned[[19, 20, 21, 22, 23, 24]].sum(axis=1)
    cleaned['female_over_64'] = cleaned[[43, 44, 45, 46, 47, 48]].sum(axis=1)
    cleaned = cleaned.rename(columns={0: 'total'})
    cols_to_drop = [19, 20, 21, 22, 23, 24, 43, 44, 45, 46, 47, 48]
    cleaned = cleaned.drop(columns=[c for c in cols_to_drop if c in cleaned.columns])
    return cleaned

over64_dfs = {2024: over64_2024, 2023: over64_2023, 2022: over64_2022, 2021: over64_2021, 2020: over64_2020}

over64_cleaned_list = []
for year, df in over64_dfs.items():
    cleaned = process_over64(df)
    cleaned = cleaned.rename(columns={
        'total': f'total_{year}',
        'male_over_64': f'male_over_64_{year}',
        'female_over_64': f'female_over_64_{year}'
    })
    over64_cleaned_list.append(cleaned)

over64_long = over64_cleaned_list[0]
for df in over64_cleaned_list[1:]:
    over64_long = over64_long.merge(df, on='censustract', how='outer')

for year in [2024, 2023, 2022, 2021, 2020]:
    total_col = f'total_{year}'
    male_col = f'male_over_64_{year}'
    female_col = f'female_over_64_{year}'
    over64_long[f'Over64_Proportion_{year}'] = round(
        (over64_long[male_col] + over64_long[female_col]) / over64_long[total_col].replace(0, np.nan) * 100, 2
    )

over64_long

,censustract,total_2024,male_over_64_2024,female_over_64_2024,total_2023,male_over_64_2023,female_over_64_2023,total_2022,male_over_64_2022,female_over_64_2022,...,male_over_64_2021,female_over_64_2021,total_2020,male_over_64_2020,female_over_64_2020,Over64_Proportion_2024,Over64_Proportion_2023,Over64_Proportion_2022,Over64_Proportion_2021,Over64_Proportion_2020
0,9001,3196,376,538,3001,406,489,3234,400,568,...,424,571,3616,388,517,28.60,29.82,29.93,28.70,25.03
1,9002.01,3117,137,108,3180,163,129,2441,92,84,...,84,66,2192,97,78,7.86,9.18,7.21,6.42,7.98
2,9002.02,3668,257,164,3520,250,144,4124,204,70,...,245,108,4254,140,201,11.48,11.19,6.64,8.27,8.02
3,9002.03,5442,158,238,4891,83,124,4589,145,119,...,88,132,4463,105,142,7.28,4.23,5.75,4.80,5.53
4,9003.01,3750,216,241,3514,136,244,3494,119,192,...,96,168,3410,80,177,12.19,10.81,8.90,7.67,7.54
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
88,9017.02,4831,213,190,4815,200,170,4641,173,188,...,134,163,4356,130,163,8.34,7.68,7.78,6.67,6.73
89,9017.03,2750,94,151,2574,103,146,2418,117,118,...,103,108,2518,105,139,8.91,9.67,9.72,7.62,9.69
90,9017.04,7269,560,312,7116,506,274,6582,476,262,...,411,249,6971,293,148,12.00,10.96,11.21,9.23,6.33
91,9019,7256,249,310,8141,277,330,7729,259,314,...,189,252,6702,238,149,7.70,7.46,7.41,6.16,5.77


In [64]:
over64_long.to_csv('Demographics/Over 64 Years Old/over64_cleaned.csv')

#### Sex Ratio

In [65]:
sex_2024 = pd.read_csv('Demographics/Sex Ratio (males per 100 females)/ACSDP5Y2024.DP05.csv')
sex_2024.loc[0, "Label (Grouping)"] = 2024
sex_2023 = pd.read_csv('Demographics/Sex Ratio (males per 100 females)/ACSDP5Y2023.DP05.csv')
sex_2023.loc[0, "Label (Grouping)"] = 2023
sex_2022 = pd.read_csv('Demographics/Sex Ratio (males per 100 females)/ACSDP5Y2022.DP05.csv')
sex_2022.loc[0, "Label (Grouping)"] = 2022   
sex_2021 = pd.read_csv('Demographics/Sex Ratio (males per 100 females)/ACSDP5Y2021.DP05.csv')
sex_2021.loc[0, "Label (Grouping)"] = 2021
sex_2020 = pd.read_csv('Demographics/Sex Ratio (males per 100 females)/ACSDP5Y2020.DP05.csv')
sex_2020.loc[0, "Label (Grouping)"] = 2020

In [67]:
def clean_header(col):
    match = re.search(r"Census Tract ([\d.]+)", col)
    return match.group(1) if match else col

def process_under5(df):
    cols_to_keep = [col for col in df.columns if '!!Estimate' in col and 'Margin of Error' not in col]
    filtered = df.iloc[[4]][cols_to_keep].copy()
    no_commas = filtered.replace(',', '', regex=True)
    numeric = no_commas.apply(pd.to_numeric, errors='coerce')
    numeric.columns = [clean_header(c) for c in numeric.columns]
    cleaned = numeric.T
    cleaned.index.name = 'censustract'
    cleaned = cleaned.reset_index()
    cleaned = cleaned.rename(columns={4: 'Sex ratio (males per 100 females)'})
    return cleaned

sex_dfs = {2024: sex_2024, 2023: sex_2023, 2022: sex_2022, 2021: sex_2021, 2020: sex_2020}

sex_cleaned_list = []
for year, df in sex_dfs.items():
    cleaned = process_under5(df)
    cleaned = cleaned.rename(columns={'Sex ratio (males per 100 females)': f'Sex ratio (males per 100 females)_{year}'})
    sex_cleaned_list.append(cleaned)

sex_ratio_long = sex_cleaned_list[0]
for df in sex_cleaned_list[1:]:
    sex_ratio_long = sex_ratio_long.merge(df, on='censustract', how='outer')

sex_ratio_long

,censustract,Sex ratio (males per 100 females)_2024,Sex ratio (males per 100 females)_2023,Sex ratio (males per 100 females)_2022,Sex ratio (males per 100 females)_2021,Sex ratio (males per 100 females)_2020
0,9001,89.7,99.0,97.6,107.1,107.5
1,9002.01,70.2,69.1,81.9,84.5,82.7
2,9002.02,109.4,119.2,118.1,112.6,109.9
3,9002.03,67.1,76.8,72.9,74.2,71.7
4,9003.01,88.3,93.4,94.4,104.9,100.8
...,...,...,...,...,...,...
88,9017.02,113.0,118.9,98.2,104.5,99.8
89,9017.03,107.1,104.8,103.2,89.3,90.8
90,9017.04,120.1,150.0,156.8,144.7,138.2
91,9019,159.0,119.1,109.2,108.8,114.4


In [68]:
sex_ratio_long.to_csv('Demographics/Sex Ratio (males per 100 females)/sex_ratio_cleaned.csv',index=False)

#### White/Black/Asian Population

In [69]:
white_2024 = pd.read_csv('Demographics/White Population/ACSDT5Y2024.B02001.csv')
white_2024.loc[0, "Label (Grouping)"] = 2024
white_2023 = pd.read_csv('Demographics/White Population/ACSDT5Y2023.B02001.csv')
white_2023.loc[0, "Label (Grouping)"] = 2023
white_2022 = pd.read_csv('Demographics/White Population/ACSDT5Y2022.B02001.csv')
white_2022.loc[0, "Label (Grouping)"] = 2022   
white_2021 = pd.read_csv('Demographics/White Population/ACSDT5Y2021.B02001.csv')
white_2021.loc[0, "Label (Grouping)"] = 2021
white_2020 = pd.read_csv('Demographics/White Population/ACSDT5Y2020.B02001.csv')
white_2020.loc[0, "Label (Grouping)"] = 2020

In [71]:
def clean_header(col):
    match = re.search(r"Census Tract ([\d.]+)", col)
    return match.group(1) if match else col

def process_under5(df):
    cols_to_keep = [col for col in df.columns if '!!Estimate' in col and 'Margin of Error' not in col]
    filtered = df.iloc[[0, 1, 2, 4]][cols_to_keep].copy()
    no_commas = filtered.replace(',', '', regex=True)
    numeric = no_commas.apply(pd.to_numeric, errors='coerce')
    numeric.columns = [clean_header(c) for c in numeric.columns]
    cleaned = numeric.T
    cleaned.index.name = 'censustract'
    cleaned = cleaned.reset_index()
    cleaned = cleaned.rename(columns={0: 'total', 1: 'white', 2: 'black', 4: 'asian'})
    return cleaned

white_dfs = {2024: white_2024, 2023: white_2023, 2022: white_2022, 2021: white_2021, 2020: white_2020}

white_cleaned_list = []
for year, df in white_dfs.items():
    cleaned = process_under5(df)
    cleaned = cleaned.rename(columns={
        'total': f'total_{year}',
        'white': f'white_{year}',
        'black': f'black_{year}',
        'asian': f'asian_{year}'
    })
    white_cleaned_list.append(cleaned)

white_long = white_cleaned_list[0]
for df in white_cleaned_list[1:]:
    white_long = white_long.merge(df, on='censustract', how='outer')

for year in [2024, 2023, 2022, 2021, 2020]:
    total_col = f'total_{year}'
    white_col = f'white_{year}'
    black_col = f'black_{year}'
    asian_col = f'asian_{year}'
    white_long[f'White_Proportion_{year}'] = round(
        white_long[white_col]/ white_long[total_col].replace(0, np.nan) * 100, 2
    )
    white_long[f'Black_Proportion_{year}'] = round(
        white_long[black_col]/ white_long[total_col].replace(0, np.nan) * 100, 2
    )
    white_long[f'Asian_Proportion_{year}'] = round(
        white_long[asian_col]/ white_long[total_col].replace(0, np.nan) * 100, 2
    )

white_long

,censustract,total_2024,white_2024,black_2024,asian_2024,total_2023,white_2023,black_2023,asian_2023,total_2022,...,Asian_Proportion_2023,White_Proportion_2022,Black_Proportion_2022,Asian_Proportion_2022,White_Proportion_2021,Black_Proportion_2021,Asian_Proportion_2021,White_Proportion_2020,Black_Proportion_2020,Asian_Proportion_2020
0,9001,3196,1742,498,391,3001,1845,388,342,3234,...,11.40,62.06,15.52,8.44,62.65,16.56,8.10,65.27,18.25,6.83
1,9002.01,3117,998,450,70,3180,1230,599,98,2441,...,3.08,41.29,23.11,3.85,44.40,24.64,7.44,51.09,27.24,5.75
2,9002.02,3668,1248,539,77,3520,1271,481,115,4124,...,3.27,33.34,13.94,2.47,37.98,15.92,1.85,41.98,11.26,1.86
3,9002.03,5442,1003,2234,197,4891,532,2213,329,4589,...,6.73,19.15,47.22,7.54,19.04,38.31,12.78,18.87,41.68,11.40
4,9003.01,3750,1933,1077,387,3514,1961,735,442,3494,...,12.58,57.30,19.00,15.11,57.32,19.20,17.05,54.90,21.14,16.72
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
88,9017.02,4831,1887,58,98,4815,1931,49,98,4641,...,2.04,51.09,1.81,2.91,49.94,1.10,3.14,52.85,1.31,3.35
89,9017.03,2750,915,167,147,2574,962,130,152,2418,...,5.91,50.66,5.17,8.19,47.36,3.54,3.68,46.47,5.72,5.32
90,9017.04,7269,3176,523,566,7116,4516,528,864,6582,...,12.14,60.89,7.49,15.74,61.39,7.13,13.51,60.02,8.81,7.49
91,9019,7256,2769,551,955,8141,4644,580,691,7729,...,8.49,56.17,13.52,11.49,66.15,9.62,11.21,64.25,15.74,7.83


In [72]:
white_long.to_csv('Demographics/White Population/Race_cleaned.csv', index=False)

#### Hispanic Population

In [73]:
hispanic_2024 = pd.read_csv('Demographics/Hispanic Population/ACSDT5Y2024.B03003.csv')
hispanic_2024.loc[0, "Label (Grouping)"] = 2024
hispanic_2023 = pd.read_csv('Demographics/Hispanic Population/ACSDT5Y2023.B03003.csv')
hispanic_2023.loc[0, "Label (Grouping)"] = 2023
hispanic_2022 = pd.read_csv('Demographics/Hispanic Population/ACSDT5Y2022.B03003.csv')
hispanic_2022.loc[0, "Label (Grouping)"] = 2022   
hispanic_2021 = pd.read_csv('Demographics/Hispanic Population/ACSDT5Y2021.B03003.csv')
hispanic_2021.loc[0, "Label (Grouping)"] = 2021
hispanic_2020 = pd.read_csv('Demographics/Hispanic Population/ACSDT5Y2020.B03003.csv')
hispanic_2020.loc[0, "Label (Grouping)"] = 2020

In [75]:
def clean_header(col):
    match = re.search(r"Census Tract ([\d.]+)", col)
    return match.group(1) if match else col

def process_under5(df):
    cols_to_keep = [col for col in df.columns if '!!Estimate' in col and 'Margin of Error' not in col]
    filtered = df.iloc[[0, 2]][cols_to_keep].copy()
    no_commas = filtered.replace(',', '', regex=True)
    numeric = no_commas.apply(pd.to_numeric, errors='coerce')
    numeric.columns = [clean_header(c) for c in numeric.columns]
    cleaned = numeric.T
    cleaned.index.name = 'censustract'
    cleaned = cleaned.reset_index()
    cleaned = cleaned.rename(columns={0: 'total', 2: 'hispanic'})
    return cleaned

hispanic_dfs = {2024: hispanic_2024, 2023: hispanic_2023, 2022: hispanic_2022, 2021: hispanic_2021, 2020: hispanic_2020}

hispanic_cleaned_list = []
for year, df in hispanic_dfs.items():
    cleaned = process_under5(df)
    cleaned = cleaned.rename(columns={
        'total': f'total_{year}',
        'hispanic': f'hispanic_{year}'
    })
    hispanic_cleaned_list.append(cleaned)

hispanic_long = hispanic_cleaned_list[0]
for df in hispanic_cleaned_list[1:]:
    hispanic_long = hispanic_long.merge(df, on='censustract', how='outer')

for year in [2024, 2023, 2022, 2021, 2020]:
    total_col = f'total_{year}'
    hispanic_col = f'hispanic_{year}'
    hispanic_long[f'Hispanic_Proportion_{year}'] = round(
        hispanic_long[hispanic_col]/ hispanic_long[total_col].replace(0, np.nan) * 100, 2
    )

hispanic_long

,censustract,total_2024,hispanic_2024,total_2023,hispanic_2023,total_2022,hispanic_2022,total_2021,hispanic_2021,total_2020,hispanic_2020,Hispanic_Proportion_2024,Hispanic_Proportion_2023,Hispanic_Proportion_2022,Hispanic_Proportion_2021,Hispanic_Proportion_2020
0,9001,3196,574,3001,560,3234,566,3467,577,3616,456,17.96,18.66,17.50,16.64,12.61
1,9002.01,3117,1644,3180,1608,2441,1104,2338,943,2192,939,52.74,50.57,45.23,40.33,42.84
2,9002.02,3668,1909,3520,1778,4124,2321,4271,2400,4254,2566,52.04,50.51,56.28,56.19,60.32
3,9002.03,5442,1999,4891,1630,4589,1355,4579,1408,4463,1382,36.73,33.33,29.53,30.75,30.97
4,9003.01,3750,696,3514,799,3494,658,3442,601,3410,632,18.56,22.74,18.83,17.46,18.53
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
88,9017.02,4831,3455,4815,3478,4641,3120,4455,3318,4356,3023,71.52,72.23,67.23,74.48,69.40
89,9017.03,2750,1656,2574,1445,2418,1328,2770,2096,2518,1792,60.22,56.14,54.92,75.67,71.17
90,9017.04,7269,4957,7116,4609,6582,4260,7151,4430,6971,4343,68.19,64.77,64.72,61.95,62.30
91,9019,7256,3199,8141,4136,7729,3791,7155,3241,6702,2949,44.09,50.80,49.05,45.30,44.00


In [76]:
hispanic_long.to_csv('Demographics/Hispanic Population/Hispanic_cleaned.csv', index=False)

#### Median Age

In [78]:
median_age_2024 = pd.read_csv('Demographics/Median Age/ACSDT5Y2024.B01002-2026-03-06T170550.csv')
median_age_2024.loc[0, "Label (Grouping)"] = 2024
median_age_2023 = pd.read_csv('Demographics/Median Age/ACSDT5Y2023.B01002-2026-03-06T170602.csv')
median_age_2023.loc[0, "Label (Grouping)"] = 2023
median_age_2022 = pd.read_csv('Demographics/Median Age/ACSDT5Y2022.B01002-2026-03-06T170614.csv')
median_age_2022.loc[0, "Label (Grouping)"] = 2022   
median_age_2021 = pd.read_csv('Demographics/Median Age/ACSDT5Y2021.B01002-2026-03-06T170623.csv')
median_age_2021.loc[0, "Label (Grouping)"] = 2021
median_age_2020 = pd.read_csv('Demographics/Median Age/ACSDT5Y2020.B01002-2026-03-06T170632.csv')
median_age_2020.loc[0, "Label (Grouping)"] = 2020

In [80]:
def clean_header(col):
    match = re.search(r"Census Tract ([\d.]+)", col)
    return match.group(1) if match else col

def process_under5(df):
    cols_to_keep = [col for col in df.columns if '!!Estimate' in col and 'Margin of Error' not in col]
    filtered = df.iloc[[1, 2, 3]][cols_to_keep].copy()
    no_commas = filtered.replace(',', '', regex=True)
    numeric = no_commas.apply(pd.to_numeric, errors='coerce')
    numeric.columns = [clean_header(c) for c in numeric.columns]
    cleaned = numeric.T
    cleaned.index.name = 'censustract'
    cleaned = cleaned.reset_index()
    cleaned = cleaned.rename(columns={1: 'median_age', 2: 'median_age_male', 3: 'median_age_female'})
    return cleaned

median_age_dfs = {2024: median_age_2024, 2023: median_age_2023, 2022: median_age_2022, 2021: median_age_2021, 2020: median_age_2020}

median_age_cleaned_list = []
for year, df in median_age_dfs.items():
    cleaned = process_under5(df)
    cleaned = cleaned.rename(columns={
        'median_age': f'median_age_{year}',
        'median_age_male': f'median_age_male_{year}',
        'median_age_female': f'median_age_female_{year}'
    })
    median_age_cleaned_list.append(cleaned)

median_age_long = median_age_cleaned_list[0]
for df in median_age_cleaned_list[1:]:
    median_age_long = median_age_long.merge(df, on='censustract', how='outer')

for year in [2024, 2023, 2022, 2021, 2020]:
    median_age_col = f'median_age_{year}'
    median_age_male_col = f'median_age_male_{year}'
    median_age_female_col = f'median_age_female_{year}'


median_age_long

,censustract,median_age_2024,median_age_male_2024,median_age_female_2024,median_age_2023,median_age_male_2023,median_age_female_2023,median_age_2022,median_age_male_2022,median_age_female_2022,median_age_2021,median_age_male_2021,median_age_female_2021,median_age_2020,median_age_male_2020,median_age_female_2020
0,9001,54.2,50.3,54.6,54.6,54.1,54.7,51.8,49.2,54.3,50.8,40.4,54.3,47.5,37.5,52.7
1,9002.01,32.7,39.3,29.1,36.3,43.5,29.3,31.9,36.2,29.6,35.1,38.2,31.2,34.8,36.7,32.7
2,9002.02,40.2,42.8,36.2,37.0,40.6,36.1,36.6,37.6,36.1,35.6,35.3,35.9,34.4,31.7,35.9
3,9002.03,30.4,31.6,30.1,29.3,29.5,29.0,29.9,33.3,26.9,29.8,33.7,24.5,29.8,33.4,26.1
4,9003.01,38.0,36.9,39.5,37.8,35.1,39.2,37.6,33.9,38.7,37.8,36.4,38.3,38.2,36.5,38.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
88,9017.02,30.0,31.0,28.2,30.8,33.0,30.5,28.7,31.0,26.4,29.9,33.1,28.4,30.6,35.1,28.4
89,9017.03,36.6,36.1,38.0,36.2,37.0,35.1,35.5,35.3,37.1,31.2,35.0,26.2,31.7,35.1,26.8
90,9017.04,32.6,34.5,31.4,32.7,33.4,32.3,33.4,35.1,32.5,31.9,33.3,31.4,29.8,30.5,28.6
91,9019,35.2,30.6,38.5,30.7,32.8,27.9,30.6,35.2,27.5,30.1,35.0,26.9,30.6,33.2,28.7


In [81]:
median_age_long.to_csv('Demographics/Median Age/Median_age_cleaned.csv', index=False)

#### Disability Rate

In [2]:
disability_2024 = pd.read_csv('Demographics/Disability Rate/ACSDT5Y2024.C18108.csv')
disability_2024.loc[0, "Label (Grouping)"] = 2024
disability_2023 = pd.read_csv('Demographics/Disability Rate/ACSDT5Y2023.C18108.csv')
disability_2023.loc[0, "Label (Grouping)"] = 2023
disability_2022 = pd.read_csv('Demographics/Disability Rate/ACSDT5Y2022.C18108.csv')
disability_2022.loc[0, "Label (Grouping)"] = 2022
disability_2021 = pd.read_csv('Demographics/Disability Rate/ACSDT5Y2021.C18108.csv')
disability_2021.loc[0, "Label (Grouping)"] = 2021
disability_2020 = pd.read_csv('Demographics/Disability Rate/ACSDT5Y2020.C18108.csv')
disability_2020.loc[0, "Label (Grouping)"] = 2020

In [7]:
def clean_header(col):
    match = re.search(r"Census Tract ([\d.]+)", col)
    return match.group(1) if match else col

def process_under5(df):
    cols_to_keep = [col for col in df.columns if '!!Estimate' in col and 'Margin of Error' not in col]
    filtered = df.iloc[[0, 4, 8, 12]][cols_to_keep].copy()
    no_commas = filtered.replace(',', '', regex=True)
    numeric = no_commas.apply(pd.to_numeric, errors='coerce')
    numeric.columns = [clean_header(c) for c in numeric.columns]
    cleaned = numeric.T
    cleaned.index.name = 'censustract'
    cleaned = cleaned.reset_index()
    cleaned = cleaned.rename(columns={0: 'total', 4: 'Nodis1', 8: 'Nodis2', 12: 'Nodis3'})
    return cleaned

nodisability_dfs = {2024: disability_2024, 2023: disability_2023, 2022: disability_2022, 2021: disability_2021, 2020: disability_2020}

nodisability_cleaned_list = []
for year, df in nodisability_dfs.items():
    cleaned = process_under5(df)
    cleaned = cleaned.rename(columns={
        'total': f'total_{year}',
        'Nodis1': f'nodis1_{year}',
        'Nodis2': f'nodis2_{year}',
        'Nodis3': f'nodis3_{year}',
    })
    nodisability_cleaned_list.append(cleaned)

disability_long = nodisability_cleaned_list[0]
for df in nodisability_cleaned_list[1:]:
    disability_long = disability_long.merge(df, on='censustract', how='outer')

for year in [2024, 2023, 2022, 2021, 2020]:
    t, n1, n2, n3 = f'total_{year}', f'nodis1_{year}', f'nodis2_{year}', f'nodis3_{year}'
    num = disability_long[t] - disability_long[n1] - disability_long[n2] - disability_long[n3]
    den = disability_long[t]
    disability_long[f'disability_rate_{year}'] = np.where(den != 0, round(num / den*100, 2), np.nan)

disability_long = disability_long[['censustract', 'disability_rate_2024', 'disability_rate_2023', 'disability_rate_2022', 'disability_rate_2021', 'disability_rate_2020']]
disability_long

,censustract,disability_rate_2024,disability_rate_2023,disability_rate_2022,disability_rate_2021,disability_rate_2020
0,9001,12.85,11.88,9.59,9.63,7.98
1,9002.01,15.65,14.17,15.44,6.30,6.14
2,9002.02,6.31,7.66,3.97,6.35,7.59
3,9002.03,12.47,10.95,10.08,9.44,9.02
4,9003.01,8.66,10.56,13.43,13.74,14.13
...,...,...,...,...,...,...
88,9017.02,13.61,13.24,11.10,10.82,7.78
89,9017.03,6.69,6.41,5.54,4.33,3.93
90,9017.04,8.54,8.27,9.01,6.46,4.29
91,9019,7.52,5.93,5.87,4.33,6.16


In [8]:
disability_long.to_csv('Demographics/Disability Rate/Disability_cleaned.csv', index=False)

#### Single Parent Households

In [9]:
household_2024 = pd.read_csv('Demographics/Single Parent Households/ACSDT5Y2020.B11003.csv')
household_2024.loc[0, "Label (Grouping)"] = 2024
household_2023 = pd.read_csv('Demographics/Single Parent Households/ACSDT5Y2023.B11003.csv')
household_2023.loc[0, "Label (Grouping)"] = 2023
household_2022 = pd.read_csv('Demographics/Single Parent Households/ACSDT5Y2022.B11003.csv')
household_2022.loc[0, "Label (Grouping)"] = 2022
household_2021 = pd.read_csv('Demographics/Single Parent Households/ACSDT5Y2021.B11003.csv')
household_2021.loc[0, "Label (Grouping)"] = 2021
household_2020 = pd.read_csv('Demographics/Single Parent Households/ACSDT5Y2020.B11003.csv')
household_2020.loc[0, "Label (Grouping)"] = 2020

In [11]:
def clean_header(col):
    match = re.search(r"Census Tract ([\d.]+)", col)
    return match.group(1) if match else col

def process_under5(df):
    cols_to_keep = [col for col in df.columns if '!!Estimate' in col and 'Margin of Error' not in col]
    filtered = df.iloc[[0, 7]][cols_to_keep].copy()
    no_commas = filtered.replace(',', '', regex=True)
    numeric = no_commas.apply(pd.to_numeric, errors='coerce')
    numeric.columns = [clean_header(c) for c in numeric.columns]
    cleaned = numeric.T
    cleaned.index.name = 'censustract'
    cleaned = cleaned.reset_index()
    cleaned = cleaned.rename(columns={0: 'total', 7: 'single_parent'})
    return cleaned

household_dfs = {2024: household_2024, 2023: household_2023, 2022: household_2022, 2021: household_2021, 2020: household_2020}

household_cleaned_list = []
for year, df in household_dfs.items():
    cleaned = process_under5(df)
    cleaned = cleaned.rename(columns={
        'total': f'total_{year}',
        'single_parent': f'single_parent_{year}',
    })
    household_cleaned_list.append(cleaned)

household_long = household_cleaned_list[0]
for df in household_cleaned_list[1:]:
    household_long = household_long.merge(df, on='censustract', how='outer')

for year in [2024, 2023, 2022, 2021, 2020]:
    t, n1 = f'total_{year}', f'single_parent_{year}'
    num = household_long[n1]
    den = household_long[t]
    household_long[f'single_parent_rate_{year}'] = np.where(den != 0, round(num / den*100, 2), np.nan)

household_long

,censustract,total_2024,single_parent_2024,total_2023,single_parent_2023,total_2022,single_parent_2022,total_2021,single_parent_2021,total_2020,single_parent_2020,single_parent_rate_2024,single_parent_rate_2023,single_parent_rate_2022,single_parent_rate_2021,single_parent_rate_2020
0,9001,1019,208,942,207,999,216,1029,229,1019,208,20.41,21.97,21.62,22.25,20.41
1,9002.01,481,137,758,214,538,199,521,133,481,137,28.48,28.23,36.99,25.53,28.48
2,9002.02,784,178,793,192,881,268,883,254,784,178,22.70,24.21,30.42,28.77,22.70
3,9002.03,1020,291,1140,460,1118,383,1071,320,1020,291,28.53,40.35,34.26,29.88,28.53
4,9003.01,744,134,791,250,833,248,782,139,744,134,18.01,31.61,29.77,17.77,18.01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
88,9017.02,1043,464,1042,406,1083,400,1126,466,1043,464,44.49,38.96,36.93,41.39,44.49
89,9017.03,515,201,558,119,529,135,528,158,515,201,39.03,21.33,25.52,29.92,39.03
90,9017.04,1208,415,1290,560,1165,482,1198,461,1208,415,34.35,43.41,41.37,38.48,34.35
91,9019,1568,523,1987,604,1926,735,1656,536,1568,523,33.35,30.40,38.16,32.37,33.35


In [12]:
household_long.to_csv('Demographics/Single Parent Households/Single_parent_cleaned.csv', index=False)

#### Group Quarters Population

In [13]:
group_2024 = pd.read_csv('Demographics/Group Quarters Population/ACSDT5Y2020.B26001.csv')
group_2024.loc[0, "Label (Grouping)"] = 2024
group_2023 = pd.read_csv('Demographics/Group Quarters Population/ACSDT5Y2023.B26001.csv')
group_2023.loc[0, "Label (Grouping)"] = 2023
group_2022 = pd.read_csv('Demographics/Group Quarters Population/ACSDT5Y2022.B26001.csv')
group_2022.loc[0, "Label (Grouping)"] = 2022
group_2021 = pd.read_csv('Demographics/Group Quarters Population/ACSDT5Y2021.B26001.csv')
group_2021.loc[0, "Label (Grouping)"] = 2021
group_2020 = pd.read_csv('Demographics/Group Quarters Population/ACSDT5Y2020.B26001.csv')
group_2020.loc[0, "Label (Grouping)"] = 2020

In [16]:
datasets = [group_2024, group_2023, group_2022, group_2021, group_2020]
group = pd.DataFrame([df.values.flatten() for df in datasets])
group.columns = group_2024.columns
group['Variable'] = 'Group Quarters Population'
group = group.rename(columns={'Label (Grouping)': 'Year'})
cols_to_keep = [col for col in group.columns if '!!Estimate' in col and 'Margin of Error' not in col]
group_filtered = group[cols_to_keep].copy()
def clean_header(col):
    match = re.search(r"Census Tract ([\d.]+)", col)
    return match.group(1) if match else col
years = [2024, 2023, 2022, 2021, 2020]
group_filtered.columns = [clean_header(c) for c in group_filtered.columns]
group_long = group_filtered.T
group_long.columns = [f'group_quarters_population_{year}' for year in years]
group_long.index.name = 'censustract'
group_long = group_long.reset_index()
group_long

,censustract,group_quarters_population_2024,group_quarters_population_2023,group_quarters_population_2022,group_quarters_population_2021,group_quarters_population_2020
0,9001,0,0,0,0,0
1,9002.01,15,5,5,12,15
2,9002.02,0,0,0,0,0
3,9002.03,8,5,6,5,8
4,9003.01,0,0,0,0,0
...,...,...,...,...,...,...
88,9017.02,0,0,0,0,0
89,9017.03,0,0,0,0,0
90,9017.04,5,5,5,5,5
91,9019,8,9,9,7,8


In [17]:
group_long.to_csv('Demographics/Group Quarters Population/Group_quarters_population_cleaned.csv', index=False)

#### Age Dependency Ratio / Old-age Dependency Ratio / Child Dependency Ratio

$\text{Child Dependency Ratio} = \frac{\text{Population age 0–14}}{\text{Population age 15–64}} \times 100$

$\text{Old-age Dependency Ratio} = \frac{\text{Population age 65+}}{\text{Population age 15–64}} \times 100$

$\text{Age Dependency Ratio} = \text{Child Dependency Ratio} + \text{Old-age Dependency Ratio}$

In [20]:
data_2024 = pd.read_csv('Demographics/Age Dependency Ratio/ACSDP5Y2024.DP05.csv')
data_2024.loc[0, "Label (Grouping)"] = 2024
data_2023 = pd.read_csv('Demographics/Age Dependency Ratio/ACSDP5Y2023.DP05.csv')
data_2023.loc[0, "Label (Grouping)"] = 2023
data_2022 = pd.read_csv('Demographics/Age Dependency Ratio/ACSDP5Y2022.DP05.csv')
data_2022.loc[0, "Label (Grouping)"] = 2022
data_2021 = pd.read_csv('Demographics/Age Dependency Ratio/ACSDP5Y2021.DP05.csv')
data_2021.loc[0, "Label (Grouping)"] = 2021
data_2020 = pd.read_csv('Demographics/Age Dependency Ratio/ACSDP5Y2020.DP05.csv')
data_2020.loc[0, "Label (Grouping)"] = 2020

In [24]:
def clean_header(col):
    match = re.search(r"Census Tract ([\d.]+)", col)
    return match.group(1) if match else col

def process_under5(df):
    cols_to_keep = [col for col in df.columns if '!!Estimate' in col and 'Margin of Error' not in col]
    filtered = df.iloc[[1, 5,6,7,8,9,10,11,12,13,14,15,16,17]][cols_to_keep].copy()
    no_commas = filtered.replace(',', '', regex=True)
    numeric = no_commas.apply(pd.to_numeric, errors='coerce')
    numeric.columns = [clean_header(c) for c in numeric.columns]
    cleaned = numeric.T
    cleaned.index.name = 'censustract'
    cleaned = cleaned.reset_index()
    cleaned = cleaned.rename(columns={1: 'total',5:'Under 5 years',6:'5 to 9 years',7:'10 to 14 years',8:'15 to 19 years',9:'20 to 24 years',10:'25 to 34 years',\
        11:'35 to 44 years',12:'45 to 54 years',13:'55 to 59 years',14:'60 to 64 years',15:'65 to 74 years',16:'75 to 84 years',17:'85 years and over'})
    return cleaned

datasets = {2024: data_2024, 2023: data_2023, 2022: data_2022, 2021: data_2021, 2020: data_2020}

data_cleaned_list = []
for year, df in datasets.items():
    cleaned = process_under5(df)
    cleaned = cleaned.rename(columns={
        'total': f'total_{year}',
        'Under 5 years': f'under_5_{year}',
        '5 to 9 years': f'5_to_9_{year}',
        '10 to 14 years': f'10_to_14_{year}',
        '15 to 19 years': f'15_to_19_{year}',
        '20 to 24 years': f'20_to_24_{year}',
        '25 to 34 years': f'25_to_34_{year}',
        '35 to 44 years': f'35_to_44_{year}',
        '45 to 54 years': f'45_to_54_{year}',
        '55 to 59 years': f'55_to_59_{year}',
        '60 to 64 years': f'60_to_64_{year}',
        '65 to 74 years': f'65_to_74_{year}',
        '75 to 84 years': f'75_to_84_{year}',
        '85 years and over': f'85_and_over_{year}',
    })
    data_cleaned_list.append(cleaned)

data_long = data_cleaned_list[0]
for df in data_cleaned_list[1:]:
    data_long = data_long.merge(df, on='censustract', how='outer')

for year in [2024, 2023, 2022, 2021, 2020]:
    t, n1, n2, n3, n4, n5, n6, n7, n8, n9, n10, n11, n12, n13 = f'total_{year}', f'under_5_{year}', f'5_to_9_{year}', f'10_to_14_{year}', f'15_to_19_{year}', f'20_to_24_{year}', f'25_to_34_{year}', f'35_to_44_{year}', f'45_to_54_{year}', f'55_to_59_{year}', f'60_to_64_{year}', f'65_to_74_{year}', f'75_to_84_{year}', f'85_and_over_{year}'
    num_015 = data_long[n1] + data_long[n2] + data_long[n3] 
    num_1564 = data_long[n4] + data_long[n5] + data_long[n6] + data_long[n7] + data_long[n8] + data_long[n9] + data_long[n10] 
    num_65 = data_long[n11] + data_long[n12] + data_long[n13]
    data_long[f'child_dependency_ratio_{year}'] = np.where(den != 0, round(num_015 / num_1564*100, 2), np.nan)
    data_long[f'old_age_dependency_ratio_{year}'] = np.where(den != 0, round(num_65 / num_1564*100, 2), np.nan)
    data_long[f'age_dependency_ratio_{year}'] = data_long[f'child_dependency_ratio_{year}'] + data_long[f'old_age_dependency_ratio_{year}']

data_long = data_long[['censustract', 'child_dependency_ratio_2024', 'child_dependency_ratio_2023', 'child_dependency_ratio_2022', 'child_dependency_ratio_2021', 'child_dependency_ratio_2020', 'old_age_dependency_ratio_2024', 'old_age_dependency_ratio_2023', 'old_age_dependency_ratio_2022', 'old_age_dependency_ratio_2021', 'old_age_dependency_ratio_2020', 'age_dependency_ratio_2024', 'age_dependency_ratio_2023', 'age_dependency_ratio_2022', 'age_dependency_ratio_2021', 'age_dependency_ratio_2020']]
data_long

,censustract,child_dependency_ratio_2024,child_dependency_ratio_2023,child_dependency_ratio_2022,child_dependency_ratio_2021,child_dependency_ratio_2020,old_age_dependency_ratio_2024,old_age_dependency_ratio_2023,old_age_dependency_ratio_2022,old_age_dependency_ratio_2021,old_age_dependency_ratio_2020,age_dependency_ratio_2024,age_dependency_ratio_2023,age_dependency_ratio_2022,age_dependency_ratio_2021,age_dependency_ratio_2020
0,9001,10.08,10.84,17.29,23.35,26.62,44.09,47.11,50.10,49.65,42.27,54.17,57.95,67.39,73.00,68.89
1,9002.01,27.47,25.78,24.45,26.40,19.56,10.87,12.72,9.67,8.67,10.37,38.34,38.50,34.12,35.07,29.93
2,9002.02,21.84,21.73,23.12,28.50,27.58,15.80,15.34,8.76,11.58,11.12,37.64,37.07,31.88,40.08,38.70
3,9002.03,45.46,55.00,50.38,47.06,45.63,11.42,6.85,9.18,7.42,8.53,56.88,61.85,59.56,54.48,54.16
4,9003.01,27.19,25.91,26.06,20.84,18.94,17.65,15.27,12.32,10.04,9.69,44.84,41.18,38.38,30.88,28.63
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
88,9017.02,35.91,33.52,36.44,27.31,23.76,12.37,11.11,11.51,9.09,8.92,48.28,44.63,47.95,36.40,32.68
89,9017.03,21.07,22.37,21.68,18.04,17.70,11.84,13.11,13.10,9.73,12.63,32.91,35.48,34.78,27.77,30.33
90,9017.04,42.76,41.27,41.16,40.01,38.52,19.46,17.39,17.83,14.24,9.36,62.22,58.66,58.99,54.25,47.88
91,9019,31.47,46.80,42.55,43.06,41.88,10.97,11.83,11.41,9.40,8.69,42.44,58.63,53.96,52.46,50.57


In [25]:
data_long.to_csv('Demographics/Age Dependency Ratio/Age_dependency_ratio_cleaned.csv', index=False)

### Socioeconomics
#### Unemployment Rate

In [31]:
unemployment_2024 = pd.read_csv('Socioeconomic/Unemployment Rate/ACSST5Y2024.S2301.csv')
unemployment_2023 = pd.read_csv('Socioeconomic/Unemployment Rate/ACSST5Y2023.S2301.csv')
unemployment_2022 = pd.read_csv('Socioeconomic/Unemployment Rate/ACSST5Y2022.S2301.csv')
unemployment_2021 = pd.read_csv('Socioeconomic/Unemployment Rate/ACSST5Y2021.S2301.csv')
unemployment_2020 = pd.read_csv('Socioeconomic/Unemployment Rate/ACSST5Y2020.S2301.csv')

In [33]:
def clean_header(col):
    match = re.search(r"Census Tract ([\d.]+)", col)
    return match.group(1) if match else col


def label_hierarchy_paths(series):
    """Build a stable full path for each ACS label row using leading-space indentation."""
    paths = []
    stack = []
    for raw in series:
        s = str(raw).replace('\xa0', ' ')
        m = re.match(r'^(\s*)', s)
        lead = m.group(1) if m else ''
        depth = len(lead) // 4
        text = s.strip()
        if not text:
            paths.append('')
            continue
        while len(stack) > depth:
            stack.pop()
        if depth == len(stack):
            stack.append(text)
        else:
            stack[depth] = text
            stack[:] = stack[: depth + 1]
        paths.append(' > '.join(stack))
    return paths


def parse_pct(val):
    if pd.isna(val):
        return np.nan
    s = str(val).strip()
    if s in ('-', '**', '', 'nan'):
        return np.nan
    s = s.replace('%', '').strip()
    return pd.to_numeric(s, errors='coerce')


def category_to_slug(cat):
    """Hyphenated slug from full label path (unique per ACS row)."""
    s = str(cat).lower().replace(' > ', '-')
    s = re.sub(r'[^a-z0-9]+', '-', s)
    s = re.sub(r'-+', '-', s).strip('-')
    return s


def process_unemployment_s2301_wide(df, year):
    """One row per censustract; columns unemployment_rate_{slug}_{year}."""
    label_col = 'Label (Grouping)'
    d = df.copy()
    d['category'] = label_hierarchy_paths(d[label_col])
    ue_cols = [
        c for c in d.columns
        if 'Census Tract' in c and '!!Unemployment rate!!Estimate' in c
    ]
    rename_map = {c: clean_header(c) for c in ue_cols}
    sub = d[['category'] + ue_cols].rename(columns=rename_map)
    long_df = sub.melt(
        id_vars=['category'],
        value_vars=list(rename_map.values()),
        var_name='censustract',
        value_name='rate',
    )
    long_df['rate'] = long_df['rate'].map(parse_pct)
    long_df = long_df[long_df['category'].astype(str).str.len() > 0]

    slug_seen = {}
    col_names = {}
    for cat in long_df['category'].unique():
        base = category_to_slug(cat)
        if base not in slug_seen:
            slug_seen[base] = 0
            key = base
        else:
            slug_seen[base] += 1
            key = f'{base}-{slug_seen[base]}'
        col_names[cat] = f'unemployment_rate_{key}_{year}'

    long_df['col'] = long_df['category'].map(col_names)
    wide = long_df.pivot_table(
        index='censustract',
        columns='col',
        values='rate',
        aggfunc='first',
        dropna=False,
    )
    wide = wide.reset_index()
    wide.columns.name = None
    return wide


unemployment_dfs = {
    2024: unemployment_2024,
    2023: unemployment_2023,
    2022: unemployment_2022,
    2021: unemployment_2021,
    2020: unemployment_2020,
}

unemployment_wide = None
for year, d in unemployment_dfs.items():
    part = process_unemployment_s2301_wide(d, year)
    if unemployment_wide is None:
        unemployment_wide = part
    else:
        unemployment_wide = unemployment_wide.merge(part, on='censustract', how='outer')

unemployment_wide

,censustract,unemployment_rate_educational-attainment-population-25-to-64-years-bachelor-s-degree-or-higher_2024,unemployment_rate_educational-attainment-population-25-to-64-years-high-school-graduate-includes-equivalency_2024,unemployment_rate_educational-attainment-population-25-to-64-years-less-than-high-school-graduate_2024,unemployment_rate_educational-attainment-population-25-to-64-years-some-college-or-associate-s-degree_2024,unemployment_rate_educational-attainment-population-25-to-64-years_2024,unemployment_rate_population-16-years-and-over-age-16-to-19-years_2024,unemployment_rate_population-16-years-and-over-age-20-to-24-years_2024,unemployment_rate_population-16-years-and-over-age-25-to-29-years_2024,unemployment_rate_population-16-years-and-over-age-30-to-34-years_2024,...,unemployment_rate_population-20-to-64-years-disability-status-with-any-disability_2020,unemployment_rate_population-20-to-64-years-poverty-status-in-the-past-12-months-at-or-above-the-poverty-level_2020,unemployment_rate_population-20-to-64-years-poverty-status-in-the-past-12-months-below-poverty-level_2020,unemployment_rate_population-20-to-64-years-sex-female-with-own-children-under-18-years-with-own-children-6-to-17-years-only_2020,unemployment_rate_population-20-to-64-years-sex-female-with-own-children-under-18-years-with-own-children-under-6-years-and-6-to-17-years_2020,unemployment_rate_population-20-to-64-years-sex-female-with-own-children-under-18-years-with-own-children-under-6-years-only_2020,unemployment_rate_population-20-to-64-years-sex-female-with-own-children-under-18-years_2020,unemployment_rate_population-20-to-64-years-sex-female_2020,unemployment_rate_population-20-to-64-years-sex-male_2020,unemployment_rate_population-20-to-64-years_2020
0,9001,2.7,19.2,0.0,0.4,3.3,0.0,25.0,0.0,0.8,...,0.0,3.5,0.0,31.3,0.0,20.5,25.6,8.5,0.0,3.4
1,9002.01,1.0,5.0,6.8,9.1,4.7,0.0,1.2,0.0,0.0,...,16.0,3.8,34.6,18.2,0.0,0.0,11.0,6.3,3.5,5.0
2,9002.02,2.4,5.3,0.0,10.2,5.1,0.0,0.0,7.7,0.0,...,19.0,4.2,0.0,2.3,25.0,0.0,8.7,4.7,3.7,4.2
3,9002.03,2.9,3.0,0.0,4.3,2.8,44.6,10.1,0.0,2.1,...,12.6,10.3,4.9,0.0,13.5,59.0,11.9,16.2,4.5,9.8
4,9003.01,0.0,0.0,0.0,13.4,4.9,83.6,4.5,1.3,22.6,...,2.0,4.1,1.7,0.0,0.0,14.3,8.8,5.5,2.5,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87,9016.02,0.0,8.7,2.1,21.4,7.9,0.0,4.0,0.0,0.0,...,22.4,5.6,9.5,8.5,0.0,36.4,13.2,5.5,6.0,5.8
88,9017.02,0.0,3.9,2.5,10.7,4.1,65.6,46.5,5.5,5.6,...,0.0,9.5,70.8,4.4,0.0,0.0,1.9,3.6,15.3,10.7
89,9017.03,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,1.2,0.0,7.6,NaN,NaN,7.6,2.4,0.0,1.1
90,9017.04,0.8,17.1,0.0,2.3,6.6,18.3,0.0,0.0,0.0,...,0.0,2.1,17.4,0.0,0.0,0.0,0.0,3.9,3.6,3.7


In [34]:
unemployment_wide.to_csv('Socioeconomic/Unemployment Rate/Unemployment_rate_cleaned.csv', index=False)

#### Less than High School Education

In [36]:
education_2024 = pd.read_csv('Socioeconomic/Less than High School Education/ACSST5Y2024.S1501-Data.csv')
education_2023 = pd.read_csv('Socioeconomic/Less than High School Education/ACSST5Y2023.S1501-Data.csv')
education_2022 = pd.read_csv('Socioeconomic/Less than High School Education/ACSST5Y2022.S1501-Data.csv')
education_2021 = pd.read_csv('Socioeconomic/Less than High School Education/ACSST5Y2021.S1501-Data.csv')
education_2020 = pd.read_csv('Socioeconomic/Less than High School Education/ACSST5Y2020.S1501-Data.csv')

In [40]:
def censustract_from_name(name):
    m = re.search(r'Census Tract ([\d.]+)', str(name))
    return m.group(1) if m else np.nan


def process_education_s1501(df, year):
    """Estimate columns only. Merge age 18-24 with 25+ into one 18+ population.

    Harmonized levels (counts sum to total 18+):
    - less_than_hs: 18-24 less than HS + 25+ less than 9th + 25+ 9th-12th no diploma
    - hs_graduate: 18-24 HS + 25+ HS
    - some_college: 18-24 some college/assoc + 25+ some college + 25+ associate
    - bachelors_plus: 18-24 bachelor+ + 25+ bachelor + 25+ graduate
    Rates are round(count / pop_total * 100, 2).
    """
    d = df.iloc[1:].copy()
    d['censustract'] = d['NAME'].map(censustract_from_name)

    def num(col):
        return pd.to_numeric(d[col], errors='coerce')

    pop_18_24 = num('S1501_C01_001E')
    pop_25p = num('S1501_C01_006E')
    pop_total = pop_18_24 + pop_25p

    n_lt_hs = num('S1501_C01_002E') + num('S1501_C01_007E') + num('S1501_C01_008E')
    n_hs = num('S1501_C01_003E') + num('S1501_C01_009E')
    n_some = num('S1501_C01_004E') + num('S1501_C01_010E') + num('S1501_C01_011E')
    n_bachp = num('S1501_C01_005E') + num('S1501_C01_012E') + num('S1501_C01_013E')

    return pd.DataFrame({
        'censustract': d['censustract'],
        f'edu_pop_total_{year}': pop_total,
        f'edu_less_than_hs_n_{year}': n_lt_hs,
        f'edu_less_than_hs_pct_{year}': np.where(
            pop_total > 0, np.round(n_lt_hs / pop_total * 100, 2), np.nan
        ),
        f'edu_hs_graduate_n_{year}': n_hs,
        f'edu_hs_graduate_pct_{year}': np.where(
            pop_total > 0, np.round(n_hs / pop_total * 100, 2), np.nan
        ),
        f'edu_some_college_n_{year}': n_some,
        f'edu_some_college_pct_{year}': np.where(
            pop_total > 0, np.round(n_some / pop_total * 100, 2), np.nan
        ),
        f'edu_bachelors_plus_n_{year}': n_bachp,
        f'edu_bachelors_plus_pct_{year}': np.where(
            pop_total > 0, np.round(n_bachp / pop_total * 100, 2), np.nan
        ),
    })


education_dfs = {
    2024: education_2024,
    2023: education_2023,
    2022: education_2022,
    2021: education_2021,
    2020: education_2020,
}

education_wide = None
for year, d in education_dfs.items():
    part = process_education_s1501(d, year)
    if education_wide is None:
        education_wide = part
    else:
        education_wide = education_wide.merge(part, on='censustract', how='outer')

education_wide.to_csv(
    'Socioeconomic/Less than High School Education/Education_attainment_cleaned.csv',
    index=False,
)
education_wide.head()

,censustract,edu_pop_total_2024,edu_less_than_hs_n_2024,edu_less_than_hs_pct_2024,edu_hs_graduate_n_2024,edu_hs_graduate_pct_2024,edu_some_college_n_2024,edu_some_college_pct_2024,edu_bachelors_plus_n_2024,edu_bachelors_plus_pct_2024,...,edu_bachelors_plus_pct_2021,edu_pop_total_2020,edu_less_than_hs_n_2020,edu_less_than_hs_pct_2020,edu_hs_graduate_n_2020,edu_hs_graduate_pct_2020,edu_some_college_n_2020,edu_some_college_pct_2020,edu_bachelors_plus_n_2020,edu_bachelors_plus_pct_2020
0,9001,2964,92,3.10,374,12.62,863,29.12,1635,55.16,...,54.39,2973,190,6.39,272,9.15,913,30.71,1598,53.75
1,9002.01,2334,444,19.02,584,25.02,477,20.44,829,35.52,...,30.63,1761,291,16.52,446,25.33,558,31.69,466,26.46
2,9002.02,2925,468,16.00,1308,44.72,726,24.82,423,14.46,...,11.81,3329,781,23.46,1288,38.69,815,24.48,445,13.37
3,9002.03,3673,893,24.31,1116,30.38,991,26.98,673,18.32,...,17.49,3083,603,19.56,966,31.33,908,29.45,606,19.66
4,9003.01,3014,476,15.79,518,17.19,970,32.18,1050,34.84,...,33.05,2786,278,9.98,502,18.02,1126,40.42,880,31.59


In [41]:
education_wide.to_csv('Socioeconomic/Less than High School Education/Education_attainment_cleaned.csv', index=False)

#### Medicaid Coverage

In [43]:
medicaid_2024 = pd.read_csv('Socioeconomic/Medicaid Coverage/ACSDT5Y2024.C27006.csv')
medicaid_2024.loc[0, "Label (Grouping)"] = 2024
medicaid_2023 = pd.read_csv('Socioeconomic/Medicaid Coverage/ACSDT5Y2023.C27006.csv')
medicaid_2023.loc[0, "Label (Grouping)"] = 2023
medicaid_2022 = pd.read_csv('Socioeconomic/Medicaid Coverage/ACSDT5Y2022.C27006.csv')
medicaid_2022.loc[0, "Label (Grouping)"] = 2022
medicaid_2021 = pd.read_csv('Socioeconomic/Medicaid Coverage/ACSDT5Y2021.C27006.csv')
medicaid_2021.loc[0, "Label (Grouping)"] = 2021
medicaid_2020 = pd.read_csv('Socioeconomic/Medicaid Coverage/ACSDT5Y2020.C27006.csv')
medicaid_2020.loc[0, "Label (Grouping)"] = 2020

In [51]:
def clean_header(col):
    match = re.search(r"Census Tract ([\d.]+)", col)
    return match.group(1) if match else col

def process_under5(df):
    cols_to_keep = [col for col in df.columns if '!!Estimate' in col and 'Margin of Error' not in col]
    filtered = df.iloc[[0, 1, 3,6,9,11, 13,16,19]][cols_to_keep].copy()
    no_commas = filtered.replace(',', '', regex=True)
    numeric = no_commas.apply(pd.to_numeric, errors='coerce')
    numeric.columns = [clean_header(c) for c in numeric.columns]
    cleaned = numeric.T
    cleaned.index.name = 'censustract'
    cleaned = cleaned.reset_index()
    cleaned = cleaned.rename(columns={0: 'total',1:'male', 3: 'male_coverage1',6:'male_coverage2',9:'male_coverage3',11:'female',13:'female_coverage1',16:'female_coverage2',19:'female_coverage3'})
    return cleaned

medicaid_dfs = {2024: medicaid_2024, 2023: medicaid_2023, 2022: medicaid_2022, 2021: medicaid_2021, 2020: medicaid_2020}

medicaid_cleaned_list = []
for year, df in medicaid_dfs.items():
    cleaned = process_under5(df)
    cleaned = cleaned.rename(columns={
        'total': f'total_{year}',
        'male': f'male_{year}',
        'male_coverage1': f'male_coverage1_{year}',
        'male_coverage2': f'male_coverage2_{year}',
        'male_coverage3': f'male_coverage3_{year}',
        'female': f'female_{year}',
        'female_coverage1': f'female_coverage1_{year}',
        'female_coverage2': f'female_coverage2_{year}',
        'female_coverage3': f'female_coverage3_{year}',
    })
    medicaid_cleaned_list.append(cleaned)

medicaid_long = medicaid_cleaned_list[0]
for df in medicaid_cleaned_list[1:]:
    medicaid_long = medicaid_long.merge(df, on='censustract', how='outer')

for year in [2024, 2023, 2022, 2021, 2020]:
    t, n1,n2,n3,n4,n5,n6,n7,n8 = f'total_{year}', f'male_{year}', f'male_coverage1_{year}', f'male_coverage2_{year}', f'male_coverage3_{year}', f'female_{year}', f'female_coverage1_{year}', f'female_coverage2_{year}', f'female_coverage3_{year}'
    num_male = medicaid_long[n2] + medicaid_long[n3] + medicaid_long[n4]
    den_male = medicaid_long[n1]
    num_female = medicaid_long[n6] + medicaid_long[n7] + medicaid_long[n8]
    den_female = medicaid_long[n5]
    medicaid_long[f'male_coverage_rate_{year}'] = np.where(den_male != 0, round(num_male / den_male*100, 2), np.nan)
    medicaid_long[f'female_coverage_rate_{year}'] = np.where(den_female != 0, round(num_female / den_female*100, 2), np.nan)
    medicaid_long[f'total_coverage_rate_{year}'] = round((medicaid_long[f'male_coverage_rate_{year}'] * medicaid_long[f'male_{year}'] + medicaid_long[f'female_coverage_rate_{year}'] * medicaid_long[f'female_{year}']) / (medicaid_long[f'male_{year}'] + medicaid_long[f'female_{year}']), 2)

medicaid_long = medicaid_long[['censustract', 'male_coverage_rate_2024', 'male_coverage_rate_2023', 'male_coverage_rate_2022', 'male_coverage_rate_2021', 'male_coverage_rate_2020', 'female_coverage_rate_2024', 'female_coverage_rate_2023', 'female_coverage_rate_2022', 'female_coverage_rate_2021', 'female_coverage_rate_2020', 'total_coverage_rate_2024', 'total_coverage_rate_2023', 'total_coverage_rate_2022', 'total_coverage_rate_2021', 'total_coverage_rate_2020']]
medicaid_long

,censustract,male_coverage_rate_2024,male_coverage_rate_2023,male_coverage_rate_2022,male_coverage_rate_2021,male_coverage_rate_2020,female_coverage_rate_2024,female_coverage_rate_2023,female_coverage_rate_2022,female_coverage_rate_2021,female_coverage_rate_2020,total_coverage_rate_2024,total_coverage_rate_2023,total_coverage_rate_2022,total_coverage_rate_2021,total_coverage_rate_2020
0,9001,25.87,28.90,27.46,25.89,22.12,33.85,33.02,33.29,31.85,27.44,30.09,31.00,30.42,28.78,24.69
1,9002.01,11.62,12.73,8.37,9.01,8.74,7.54,8.45,8.91,6.27,8.30,9.21,10.19,8.67,7.51,8.50
2,9002.02,15.32,13.68,8.16,9.80,5.83,10.33,9.59,5.02,7.12,11.45,12.90,11.77,6.71,8.54,8.52
3,9002.03,8.43,4.31,8.91,6.05,6.98,7.22,4.05,4.41,4.30,3.92,7.70,4.16,6.30,5.04,5.19
4,9003.01,12.19,8.13,6.01,4.11,3.34,11.95,13.87,11.35,10.60,11.01,12.06,11.14,8.78,7.31,7.20
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
88,9017.02,7.74,7.59,7.02,5.98,5.97,9.35,8.64,9.22,7.99,8.85,8.50,8.07,8.14,6.97,7.41
89,9017.03,7.31,7.21,9.12,9.49,15.03,9.11,8.99,9.16,6.43,14.47,8.18,8.08,9.14,7.87,14.74
90,9017.04,13.77,10.56,10.65,9.86,7.37,8.79,8.96,8.82,7.87,5.88,11.51,9.92,9.94,9.05,6.74
91,9019,5.57,5.83,6.17,5.34,6.91,10.17,7.54,8.46,7.59,5.75,7.34,6.61,7.26,6.42,6.37


In [52]:
medicaid_long.to_csv('Socioeconomic/Medicaid Coverage/Medicaid_coverage_cleaned.csv', index=False)

#### Social Security Income

In [55]:
ssi_2024 = pd.read_csv('Socioeconomic/Social Security Income/ACSDT5Y2024.B19055.csv')
ssi_2024.loc[0, "Label (Grouping)"] = 2024
ssi_2023 = pd.read_csv('Socioeconomic/Social Security Income/ACSDT5Y2023.B19055.csv')
ssi_2023.loc[0, "Label (Grouping)"] = 2023
ssi_2022 = pd.read_csv('Socioeconomic/Social Security Income/ACSDT5Y2022.B19055.csv')
ssi_2022.loc[0, "Label (Grouping)"] = 2022
ssi_2021 = pd.read_csv('Socioeconomic/Social Security Income/ACSDT5Y2021.B19055.csv')
ssi_2021.loc[0, "Label (Grouping)"] = 2021
ssi_2020 = pd.read_csv('Socioeconomic/Social Security Income/ACSDT5Y2020.B19055.csv')
ssi_2020.loc[0, "Label (Grouping)"] = 2020

In [59]:
def clean_header(col):
    match = re.search(r"Census Tract ([\d.]+)", col)
    return match.group(1) if match else col

def process_under5(df):
    cols_to_keep = [col for col in df.columns if '!!Estimate' in col and 'Margin of Error' not in col]
    filtered = df.iloc[[0, 1]][cols_to_keep].copy()
    no_commas = filtered.replace(',', '', regex=True)
    numeric = no_commas.apply(pd.to_numeric, errors='coerce')
    numeric.columns = [clean_header(c) for c in numeric.columns]
    cleaned = numeric.T
    cleaned.index.name = 'censustract'
    cleaned = cleaned.reset_index()
    cleaned = cleaned.rename(columns={0: 'total',1:'With SSI'})
    return cleaned

ssi_dfs = {2024: ssi_2024, 2023: ssi_2023, 2022: ssi_2022, 2021: ssi_2021, 2020: ssi_2020}

ssi_cleaned_list = []
for year, df in ssi_dfs.items():
    cleaned = process_under5(df)
    cleaned = cleaned.rename(columns={
        'total': f'total_{year}',
        'With SSI': f'With_SSI_{year}',
    })
    ssi_cleaned_list.append(cleaned)

ssi_long = ssi_cleaned_list[0]
for df in ssi_cleaned_list[1:]:
    ssi_long = ssi_long.merge(df, on='censustract', how='outer')

for year in [2024, 2023, 2022, 2021, 2020]:
    t, n1 = f'total_{year}', f'With_SSI_{year}'
    prop_SSI = ssi_long[n1] / ssi_long[t]
    ssi_long[f'prop_SSI_{year}'] = round(prop_SSI*100, 2)

ssi_long

,censustract,total_2024,With_SSI_2024,total_2023,With_SSI_2023,total_2022,With_SSI_2022,total_2021,With_SSI_2021,total_2020,With_SSI_2020,prop_SSI_2024,prop_SSI_2023,prop_SSI_2022,prop_SSI_2021,prop_SSI_2020
0,9001,1627,585,1536,562,1559,606,1550,641,1462,596,35.96,36.59,38.87,41.35,40.77
1,9002.01,1051,223,1091,251,961,152,880,137,772,132,21.22,23.01,15.82,15.57,17.10
2,9002.02,1075,258,1055,220,1125,117,1166,155,1130,206,24.00,20.85,10.40,13.29,18.23
3,9002.03,1490,279,1387,158,1406,199,1419,173,1413,181,18.72,11.39,14.15,12.19,12.81
4,9003.01,1599,351,1538,303,1503,234,1513,198,1465,194,21.95,19.70,15.57,13.09,13.24
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
88,9017.02,1256,274,1233,281,1243,266,1263,248,1198,237,21.82,22.79,21.40,19.64,19.78
89,9017.03,740,125,703,121,672,134,650,134,642,194,16.89,17.21,19.94,20.62,30.22
90,9017.04,1942,466,1969,349,1937,373,1946,420,1905,326,24.00,17.72,19.26,21.58,17.11
91,9019,2385,329,2351,286,2317,250,2153,249,2117,265,13.79,12.17,10.79,11.57,12.52


In [60]:
ssi_long.to_csv('Socioeconomic/Social Security Income/SSI_cleaned.csv', index=False)

#### Food Stamps/SNAP 

In [65]:
snap_2024 = pd.read_csv('Socioeconomic/Food Stamps:SNAP/ACSDT5Y2024.B22010.csv')
snap_2023 = pd.read_csv('Socioeconomic/Food Stamps:SNAP/ACSDT5Y2023.B22010.csv')
snap_2022 = pd.read_csv('Socioeconomic/Food Stamps:SNAP/ACSDT5Y2022.B22010.csv')
snap_2021 = pd.read_csv('Socioeconomic/Food Stamps:SNAP/ACSDT5Y2021.B22010.csv')
snap_2020 = pd.read_csv('Socioeconomic/Food Stamps:SNAP/ACSDT5Y2020.B22010.csv')

In [67]:
def clean_header(col):
    match = re.search(r"Census Tract ([\d.]+)", col)
    return match.group(1) if match else col

def process_under5(df):
    cols_to_keep = [col for col in df.columns if '!!Estimate' in col and 'Margin of Error' not in col]
    filtered = df.iloc[[0, 1]][cols_to_keep].copy()
    no_commas = filtered.replace(',', '', regex=True)
    numeric = no_commas.apply(pd.to_numeric, errors='coerce')
    numeric.columns = [clean_header(c) for c in numeric.columns]
    cleaned = numeric.T
    cleaned.index.name = 'censustract'
    cleaned = cleaned.reset_index()
    cleaned = cleaned.rename(columns={0: 'total',1:'With SNAP'})
    return cleaned

snap_dfs = {2024: snap_2024, 2023: snap_2023, 2022: snap_2022, 2021: snap_2021, 2020: snap_2020}

snap_cleaned_list = []
for year, df in snap_dfs.items():
    cleaned = process_under5(df)
    cleaned = cleaned.rename(columns={
        'total': f'total_{year}',
        'With SNAP': f'With_SNAP_{year}',
    })
    snap_cleaned_list.append(cleaned)

snap_long = snap_cleaned_list[0]
for df in snap_cleaned_list[1:]:
    snap_long = snap_long.merge(df, on='censustract', how='outer')

for year in [2024, 2023, 2022, 2021, 2020]:
    t, n1 = f'total_{year}', f'With_SNAP_{year}'
    prop_SNAP = snap_long[n1] / snap_long[t]
    snap_long[f'prop_SNAP_{year}'] = round(prop_SNAP*100, 2)

snap_long

,censustract,total_2024,With_SNAP_2024,total_2023,With_SNAP_2023,total_2022,With_SNAP_2022,total_2021,With_SNAP_2021,total_2020,With_SNAP_2020,prop_SNAP_2024,prop_SNAP_2023,prop_SNAP_2022,prop_SNAP_2021,prop_SNAP_2020
0,9001,1627,12,1536,11,1559,23,1550,10,1462,7,0.74,0.72,1.48,0.65,0.48
1,9002.01,1051,94,1091,67,961,77,880,49,772,12,8.94,6.14,8.01,5.57,1.55
2,9002.02,1075,117,1055,138,1125,129,1166,164,1130,81,10.88,13.08,11.47,14.07,7.17
3,9002.03,1490,115,1387,193,1406,107,1419,101,1413,78,7.72,13.91,7.61,7.12,5.52
4,9003.01,1599,99,1538,84,1503,130,1513,137,1465,98,6.19,5.46,8.65,9.05,6.69
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
88,9017.02,1256,88,1233,94,1243,68,1263,43,1198,57,7.01,7.62,5.47,3.40,4.76
89,9017.03,740,55,703,62,672,47,650,11,642,42,7.43,8.82,6.99,1.69,6.54
90,9017.04,1942,275,1969,213,1937,147,1946,176,1905,111,14.16,10.82,7.59,9.04,5.83
91,9019,2385,268,2351,164,2317,110,2153,151,2117,200,11.24,6.98,4.75,7.01,9.45


In [68]:
snap_long.to_csv('Socioeconomic/Food Stamps:SNAP/SNAP_cleaned.csv', index=False)

#### Limited English Proficiency

In [71]:
english_2024 = pd.read_csv('Socioeconomic/Limited English Proficiency/ACSDT5Y2024.C16001.csv')
english_2024.loc[0, "Label (Grouping)"] = 2024
english_2023 = pd.read_csv('Socioeconomic/Limited English Proficiency/ACSDT5Y2023.C16001.csv')
english_2023.loc[0, "Label (Grouping)"] = 2023
english_2022 = pd.read_csv('Socioeconomic/Limited English Proficiency/ACSDT5Y2022.C16001.csv')
english_2022.loc[0, "Label (Grouping)"] = 2022
english_2021 = pd.read_csv('Socioeconomic/Limited English Proficiency/ACSDT5Y2021.C16001.csv')
english_2021.loc[0, "Label (Grouping)"] = 2021
english_2020 = pd.read_csv('Socioeconomic/Limited English Proficiency/ACSDT5Y2020.C16001.csv')
english_2020.loc[0, "Label (Grouping)"] = 2020

In [73]:
def clean_header(col):
    match = re.search(r"Census Tract ([\d.]+)", col)
    return match.group(1) if match else col

def process_under5(df):
    cols_to_keep = [col for col in df.columns if '!!Estimate' in col and 'Margin of Error' not in col]
    filtered = df.iloc[[0, 4,7,10,13,16,19,22,25,28,31,34,37]][cols_to_keep].copy()
    no_commas = filtered.replace(',', '', regex=True)
    numeric = no_commas.apply(pd.to_numeric, errors='coerce')
    numeric.columns = [clean_header(c) for c in numeric.columns]
    cleaned = numeric.T
    cleaned.index.name = 'censustract'
    cleaned = cleaned.reset_index()
    cleaned = cleaned.rename(columns={0: 'total',4:'Limited_English_Proficiency_1',7:'Limited_English_Proficiency_2',10:'Limited_English_Proficiency_3',13:'Limited_English_Proficiency_4',16:'Limited_English_Proficiency_5',19:'Limited_English_Proficiency_6',22:'Limited_English_Proficiency_7',25:'Limited_English_Proficiency_8',28:'Limited_English_Proficiency_9',31:'Limited_English_Proficiency_10',34:'Limited_English_Proficiency_11',37:'Limited_English_Proficiency_12'})
    return cleaned

english_dfs = {2024: english_2024, 2023: english_2023, 2022: english_2022, 2021: english_2021, 2020: english_2020}

english_cleaned_list = []
for year, df in english_dfs.items():
    cleaned = process_under5(df)
    cleaned = cleaned.rename(columns={
        'total': f'total_{year}',
        'Limited_English_Proficiency_1': f'Limited_English_Proficiency_1_{year}',
        'Limited_English_Proficiency_2': f'Limited_English_Proficiency_2_{year}',
        'Limited_English_Proficiency_3': f'Limited_English_Proficiency_3_{year}',
        'Limited_English_Proficiency_4': f'Limited_English_Proficiency_4_{year}',
        'Limited_English_Proficiency_5': f'Limited_English_Proficiency_5_{year}',
        'Limited_English_Proficiency_6': f'Limited_English_Proficiency_6_{year}',
        'Limited_English_Proficiency_7': f'Limited_English_Proficiency_7_{year}',
        'Limited_English_Proficiency_8': f'Limited_English_Proficiency_8_{year}',
        'Limited_English_Proficiency_9': f'Limited_English_Proficiency_9_{year}',
        'Limited_English_Proficiency_10': f'Limited_English_Proficiency_10_{year}',
        'Limited_English_Proficiency_11': f'Limited_English_Proficiency_11_{year}',
        'Limited_English_Proficiency_12': f'Limited_English_Proficiency_12_{year}',
    })
    english_cleaned_list.append(cleaned)

english_long = english_cleaned_list[0]
for df in english_cleaned_list[1:]:
    english_long = english_long.merge(df, on='censustract', how='outer')

for year in [2024, 2023, 2022, 2021, 2020]:
    t = f'total_{year}'
    parts = [f'Limited_English_Proficiency_{i}_{year}' for i in range(1, 13)]
    num = sum(english_long[c] for c in parts)
    den = english_long[t]
    english_long[f'prop_limited_english_{year}'] = np.where(
        den != 0, np.round(num / den * 100, 2), np.nan
    )

english_long = english_long[['censustract', 'prop_limited_english_2024', 'prop_limited_english_2023', 'prop_limited_english_2022', 'prop_limited_english_2021', 'prop_limited_english_2020']]
english_long

,censustract,prop_limited_english_2024,prop_limited_english_2023,prop_limited_english_2022,prop_limited_english_2021,prop_limited_english_2020
0,9001,4.83,8.11,8.76,7.26,5.75
1,9002.01,30.94,26.78,20.62,18.53,23.18
2,9002.02,32.25,29.30,34.54,33.05,38.39
3,9002.03,21.60,21.43,19.34,22.20,23.95
4,9003.01,13.26,12.96,13.59,11.22,11.72
...,...,...,...,...,...,...
88,9017.02,38.67,40.96,33.46,40.56,37.62
89,9017.03,34.43,32.32,26.88,29.05,25.69
90,9017.04,48.19,48.03,48.87,44.93,39.34
91,9019,19.09,18.78,21.52,20.07,20.41


In [74]:
english_long.to_csv('Socioeconomic/Limited English Proficiency/Limited_English_Proficiency_cleaned.csv', index=False)

#### Uninsured Population

In [ ]:
uninsured_2024 = pd.read_csv('Socioeconomic/Uninsured Population/ACSST5Y2024.S2701.csv')
uninsured_2023 = pd.read_csv('Socioeconomic/Uninsured Population/ACSST5Y2023.S2701.csv')
uninsured_2022 = pd.read_csv('Socioeconomic/Uninsured Population/ACSST5Y2022.S2701.csv')
uninsured_2021 = pd.read_csv('Socioeconomic/Uninsured Population/ACSST5Y2021.S2701.csv')
uninsured_2020 = pd.read_csv('Socioeconomic/Uninsured Population/ACSST5Y2020.S2701.csv')

In [77]:
from collections import defaultdict


def clean_header(col):
    match = re.search(r"Census Tract ([\d.]+)", col)
    return match.group(1) if match else None


def parse_pct_uninsured(val):
    if pd.isna(val):
        return np.nan
    s = str(val).strip()
    if s in ('-', '**', ''):
        return np.nan
    s = s.replace('%', '').strip()
    return pd.to_numeric(s, errors='coerce')


def parse_num_uninsured(val):
    if pd.isna(val):
        return np.nan
    return pd.to_numeric(str(val).replace(',', '').strip(), errors='coerce')


def process_uninsured_year(df, year):
    """Estimate columns only. Rows: 0 total pop, 15 male, 16 female (S2701)."""
    cols = [
        c
        for c in df.columns
        if '!!Estimate' in c
        and 'Margin of Error' not in c
        and 'Census Tract' in c
    ]
    by_tract = defaultdict(list)
    for c in cols:
        h = clean_header(c)
        if h:
            by_tract[h].append(c)

    def metrics_for_row(row_idx):
        out = {}
        for tract, tcols in by_tract.items():
            pop_c = next(x for x in tcols if x.endswith('!!Total!!Estimate'))
            uni_c = next(
                x
                for x in tcols
                if '!!Uninsured!!Estimate' in x and 'Percent' not in x
            )
            pct_c = next(x for x in tcols if '!!Percent Uninsured!!Estimate' in x)
            out[tract] = {
                'pop': parse_num_uninsured(df.iloc[row_idx][pop_c]),
                'uninsured_n': parse_num_uninsured(df.iloc[row_idx][uni_c]),
                'uninsured_pct': parse_pct_uninsured(df.iloc[row_idx][pct_c]),
            }
        return out

    m0 = metrics_for_row(0)
    m15 = metrics_for_row(15)
    m16 = metrics_for_row(16)
    tracts = sorted(m0.keys())
    rows = []
    for t in tracts:
        rows.append(
            {
                'censustract': t,
                f'pop_total_{year}': m0[t]['pop'],
                f'uninsured_n_total_{year}': m0[t]['uninsured_n'],
                f'uninsured_pct_total_{year}': np.round(m0[t]['uninsured_pct'], 2),
                f'pop_male_{year}': m15[t]['pop'],
                f'uninsured_n_male_{year}': m15[t]['uninsured_n'],
                f'uninsured_pct_male_{year}': np.round(m15[t]['uninsured_pct'], 2),
                f'pop_female_{year}': m16[t]['pop'],
                f'uninsured_n_female_{year}': m16[t]['uninsured_n'],
                f'uninsured_pct_female_{year}': np.round(m16[t]['uninsured_pct'], 2),
            }
        )
    return pd.DataFrame(rows)


uninsured_dfs = {
    2024: uninsured_2024,
    2023: uninsured_2023,
    2022: uninsured_2022,
    2021: uninsured_2021,
    2020: uninsured_2020,
}

uninsured_wide = None
for year, d in uninsured_dfs.items():
    part = process_uninsured_year(d, year)
    if uninsured_wide is None:
        uninsured_wide = part
    else:
        uninsured_wide = uninsured_wide.merge(part, on='censustract', how='outer')

uninsured_wide.to_csv(
    'Socioeconomic/Uninsured Population/Uninsured_population_cleaned.csv', index=False
)
uninsured_wide.head()

,censustract,pop_total_2024,uninsured_n_total_2024,uninsured_pct_total_2024,pop_male_2024,uninsured_n_male_2024,uninsured_pct_male_2024,pop_female_2024,uninsured_n_female_2024,uninsured_pct_female_2024,...,uninsured_pct_female_2021,pop_total_2020,uninsured_n_total_2020,uninsured_pct_total_2020,pop_male_2020,uninsured_n_male_2020,uninsured_pct_male_2020,pop_female_2020,uninsured_n_female_2020,uninsured_pct_female_2020
0,9001,3160.0,172.0,5.4,1488.0,123.0,8.3,1672.0,49.0,2.9,...,1.9,3535.0,317.0,9.0,1826.0,217.0,11.9,1709.0,100.0,5.9
1,9002.01,3105.0,828.0,26.7,1274.0,302.0,23.7,1831.0,526.0,28.7,...,5.0,2165.0,87.0,4.0,972.0,42.0,4.3,1193.0,45.0,3.8
2,9002.02,3612.0,601.0,16.6,1860.0,389.0,20.9,1752.0,212.0,12.1,...,11.1,4241.0,755.0,17.8,2214.0,431.0,19.5,2027.0,324.0,16.0
3,9002.03,5427.0,560.0,10.3,2171.0,208.0,9.6,3256.0,352.0,10.8,...,21.0,4447.0,833.0,18.7,1848.0,374.0,20.2,2599.0,459.0,17.7
4,9003.01,3706.0,389.0,10.5,1714.0,216.0,12.6,1992.0,173.0,8.7,...,4.6,3375.0,201.0,6.0,1677.0,102.0,6.1,1698.0,99.0,5.8


#### Low Income Population

In [78]:
lowincome_2024 = pd.read_csv('Socioeconomic/Low Income Population/ACSDT5Y2024.B19001.csv')
lowincome_2023 = pd.read_csv('Socioeconomic/Low Income Population/ACSDT5Y2023.B19001.csv')
lowincome_2022 = pd.read_csv('Socioeconomic/Low Income Population/ACSDT5Y2022.B19001.csv')
lowincome_2021 = pd.read_csv('Socioeconomic/Low Income Population/ACSDT5Y2021.B19001.csv')
lowincome_2020 = pd.read_csv('Socioeconomic/Low Income Population/ACSDT5Y2020.B19001.csv')

In [80]:
def clean_header(col):
    match = re.search(r"Census Tract ([\d.]+)", col)
    return match.group(1) if match else None


BRACKET_KEYS = [
    'hh_total',
    'lt_10k',
    '10k_15k',
    '15k_20k',
    '20k_25k',
    '25k_30k',
    '30k_35k',
    '35k_40k',
    '40k_45k',
    '45k_50k',
    '50k_60k',
    '60k_75k',
    '75k_100k',
    '100k_125k',
    '125k_150k',
    '150k_200k',
    '200k_plus',
]


def process_b19001_year(df, year):
    """Tract x row; estimate only. Rows 0–16 = total + 16 ACS income brackets."""
    cols = [
        c
        for c in df.columns
        if '!!Estimate' in c
        and 'Margin of Error' not in c
        and 'Census Tract' in c
    ]
    filtered = df.iloc[0:17][cols].copy()
    filtered = filtered.replace(',', '', regex=True)
    numeric = filtered.apply(pd.to_numeric, errors='coerce')
    numeric.columns = [clean_header(c) for c in numeric.columns]
    cleaned = numeric.T
    cleaned.index.name = 'censustract'
    cleaned = cleaned.reset_index()
    cleaned = cleaned.rename(columns={i: BRACKET_KEYS[i] for i in range(17)})

    low_n = (
        cleaned['lt_10k']
        + cleaned['10k_15k']
        + cleaned['15k_20k']
        + cleaned['20k_25k']
        + cleaned['25k_30k']
        + cleaned['30k_35k']
    )
    tot = cleaned['hh_total']
    cleaned[f'low_income_lt35k_pct_{year}'] = np.where(
        tot > 0, np.round(low_n / tot * 100, 2), np.nan
    )

    rename = {'censustract': 'censustract'}
    for k in BRACKET_KEYS:
        rename[k] = f'{k}_{year}'
    cleaned = cleaned.rename(columns=rename)
    return cleaned


lowincome_dfs = {
    2024: lowincome_2024,
    2023: lowincome_2023,
    2022: lowincome_2022,
    2021: lowincome_2021,
    2020: lowincome_2020,
}

lowincome_wide = None
for year, d in lowincome_dfs.items():
    part = process_b19001_year(d, year)
    if lowincome_wide is None:
        lowincome_wide = part
    else:
        lowincome_wide = lowincome_wide.merge(part, on='censustract', how='outer')

lowincome_wide.to_csv(
    'Socioeconomic/Low Income Population/Low_income_households_cleaned.csv', index=False
)
lowincome_wide.head()

,censustract,hh_total_2024,lt_10k_2024,10k_15k_2024,15k_20k_2024,20k_25k_2024,25k_30k_2024,30k_35k_2024,35k_40k_2024,40k_45k_2024,...,40k_45k_2020,45k_50k_2020,50k_60k_2020,60k_75k_2020,75k_100k_2020,100k_125k_2020,125k_150k_2020,150k_200k_2020,200k_plus_2020,low_income_lt35k_pct_2020
0,9001,1627,26,0,0,12,9,14,17,2,...,29,47,43,59,95,253,231,282,261,9.78
1,9002.01,1051,55,27,0,26,0,22,20,19,...,31,30,70,49,115,77,87,93,67,14.90
2,9002.02,1075,0,14,0,0,24,24,14,0,...,13,50,94,121,201,264,93,121,33,11.95
3,9002.03,1490,0,61,0,11,83,12,12,68,...,129,35,87,196,125,300,138,27,14,23.14
4,9003.01,1599,58,55,13,0,103,44,66,62,...,20,39,62,229,285,127,50,168,153,15.02


#### Population Below 150% Poverty Level

In [82]:
poverty_2024 = pd.read_csv('Socioeconomic/Population Below 150% Poverty Level/ACSDT5Y2024.C17002.csv')
poverty_2023 = pd.read_csv('Socioeconomic/Population Below 150% Poverty Level/ACSDT5Y2023.C17002.csv')
poverty_2022 = pd.read_csv('Socioeconomic/Population Below 150% Poverty Level/ACSDT5Y2022.C17002.csv')
poverty_2021 = pd.read_csv('Socioeconomic/Population Below 150% Poverty Level/ACSDT5Y2021.C17002.csv')
poverty_2020 = pd.read_csv('Socioeconomic/Population Below 150% Poverty Level/ACSDT5Y2020.C17002.csv')

In [84]:
def clean_header(col):
    match = re.search(r"Census Tract ([\d.]+)", col)
    return match.group(1) if match else col

def process_under5(df):
    cols_to_keep = [col for col in df.columns if '!!Estimate' in col and 'Margin of Error' not in col]
    filtered = df.iloc[[0, 1, 2,3,4]][cols_to_keep].copy()
    no_commas = filtered.replace(',', '', regex=True)
    numeric = no_commas.apply(pd.to_numeric, errors='coerce')
    numeric.columns = [clean_header(c) for c in numeric.columns]
    cleaned = numeric.T
    cleaned.index.name = 'censustract'
    cleaned = cleaned.reset_index()
    cleaned = cleaned.rename(columns={0: 'total',1:'1', 2:'2',3:'3',4:'4'})
    return cleaned

poverty_dfs = {2024: poverty_2024, 2023: poverty_2023, 2022: poverty_2022, 2021: poverty_2021, 2020: poverty_2020}

poverty_cleaned_list = []
for year, df in poverty_dfs.items():
    cleaned = process_under5(df)
    cleaned = cleaned.rename(columns={
        'total': f'total_{year}',
        '1': f'1_{year}',
        '2': f'2_{year}',
        '3': f'3_{year}',
        '4': f'4_{year}',
    })
    poverty_cleaned_list.append(cleaned)

poverty_long = poverty_cleaned_list[0]
for df in poverty_cleaned_list[1:]:
    poverty_long = poverty_long.merge(df, on='censustract', how='outer')

for year in [2024, 2023, 2022, 2021, 2020]:
    t, n1, n2, n3, n4 = f'total_{year}', f'1_{year}', f'2_{year}', f'3_{year}', f'4_{year}'
    prop_under150poverty = (poverty_long[n1]+poverty_long[n2]+poverty_long[n3]+poverty_long[n4]) / poverty_long[t]
    poverty_long[f'prop_under150poverty_{year}'] = round(prop_under150poverty*100, 2)

poverty_long = poverty_long[['censustract', 'prop_under150poverty_2024', 'prop_under150poverty_2023', 'prop_under150poverty_2022', 'prop_under150poverty_2021', 'prop_under150poverty_2020']] 
poverty_long   

,censustract,prop_under150poverty_2024,prop_under150poverty_2023,prop_under150poverty_2022,prop_under150poverty_2021,prop_under150poverty_2020
0,9001,2.32,2.70,3.15,3.63,4.59
1,9002.01,34.46,25.48,25.62,17.50,15.66
2,9002.02,5.13,4.55,3.03,2.90,3.23
3,9002.03,26.98,34.74,23.01,24.72,21.09
4,9003.01,20.88,22.31,20.95,19.67,17.92
...,...,...,...,...,...,...
88,9017.02,17.06,14.55,17.12,9.96,10.90
89,9017.03,7.74,5.08,3.33,8.60,7.68
90,9017.04,17.92,16.89,17.47,18.61,26.30
91,9019,23.65,15.91,17.19,26.53,25.68


In [85]:
poverty_long.to_csv('Socioeconomic/Population Below 150% Poverty Level/Population_below_150poverty_cleaned.csv', index=False)

#### Public Assistance Income

In [86]:
public_assistance_2024 = pd.read_csv('Socioeconomic/Public Assistance Income/ACSDT5Y2024.B19057.csv')
public_assistance_2023 = pd.read_csv('Socioeconomic/Public Assistance Income/ACSDT5Y2023.B19057.csv')
public_assistance_2022 = pd.read_csv('Socioeconomic/Public Assistance Income/ACSDT5Y2022.B19057.csv')
public_assistance_2021 = pd.read_csv('Socioeconomic/Public Assistance Income/ACSDT5Y2021.B19057.csv')
public_assistance_2020 = pd.read_csv('Socioeconomic/Public Assistance Income/ACSDT5Y2020.B19057.csv')


In [88]:
def clean_header(col):
    match = re.search(r"Census Tract ([\d.]+)", col)
    return match.group(1) if match else col

def process_under5(df):
    cols_to_keep = [col for col in df.columns if '!!Estimate' in col and 'Margin of Error' not in col]
    filtered = df.iloc[[0, 1]][cols_to_keep].copy()
    no_commas = filtered.replace(',', '', regex=True)
    numeric = no_commas.apply(pd.to_numeric, errors='coerce')
    numeric.columns = [clean_header(c) for c in numeric.columns]
    cleaned = numeric.T
    cleaned.index.name = 'censustract'
    cleaned = cleaned.reset_index()
    cleaned = cleaned.rename(columns={0: 'total',1:'With Public Assistance'})
    return cleaned

public_assistance_dfs = {2024: public_assistance_2024, 2023: public_assistance_2023, 2022: public_assistance_2022, 2021: public_assistance_2021, 2020: public_assistance_2020}

public_assistance_cleaned_list = []
for year, df in public_assistance_dfs.items():
    cleaned = process_under5(df)
    cleaned = cleaned.rename(columns={
        'total': f'total_{year}',
        'With Public Assistance': f'With_Public_Assistance_{year}',
    })
    public_assistance_cleaned_list.append(cleaned)

public_assistance_long = public_assistance_cleaned_list[0]
for df in public_assistance_cleaned_list[1:]:
    public_assistance_long = public_assistance_long.merge(df, on='censustract', how='outer')

for year in [2024, 2023, 2022, 2021, 2020]:
    t, n1 = f'total_{year}', f'With_Public_Assistance_{year}'
    prop_with_public_assistance = public_assistance_long[n1] / public_assistance_long[t]
    public_assistance_long[f'prop_with_public_assistance_{year}'] = round(prop_with_public_assistance*100, 2)

public_assistance_long = public_assistance_long[['censustract', 'prop_with_public_assistance_2024', 'prop_with_public_assistance_2023', 'prop_with_public_assistance_2022', 'prop_with_public_assistance_2021', 'prop_with_public_assistance_2020']] 
public_assistance_long  

,censustract,prop_with_public_assistance_2024,prop_with_public_assistance_2023,prop_with_public_assistance_2022,prop_with_public_assistance_2021,prop_with_public_assistance_2020
0,9001,0.61,0.59,0.58,0.58,0.48
1,9002.01,0.00,0.00,0.00,0.00,0.00
2,9002.02,0.65,0.47,1.96,1.54,0.71
3,9002.03,0.00,0.00,0.00,0.00,0.57
4,9003.01,1.00,0.46,2.86,3.97,3.96
...,...,...,...,...,...,...
88,9017.02,1.27,1.87,2.25,0.48,1.09
89,9017.03,0.68,1.71,0.89,0.00,0.00
90,9017.04,0.57,0.51,0.00,0.00,0.00
91,9019,0.00,0.00,0.00,0.51,0.61


In [89]:
public_assistance_long.to_csv('Socioeconomic/Public Assistance Income/Public_assistance_income_cleaned.csv', index=False)

### Housing
#### Homeownership Rate

In [91]:
homeownership_2024 = pd.read_csv('Housing/Homeownership Rate/ACSDT5Y2024.B25008.csv')
homeownership_2023 = pd.read_csv('Housing/Homeownership Rate/ACSDT5Y2023.B25008.csv')
homeownership_2022 = pd.read_csv('Housing/Homeownership Rate/ACSDT5Y2022.B25008.csv')
homeownership_2021 = pd.read_csv('Housing/Homeownership Rate/ACSDT5Y2021.B25008.csv')
homeownership_2020 = pd.read_csv('Housing/Homeownership Rate/ACSDT5Y2020.B25008.csv')

In [97]:
def clean_header(col):
    match = re.search(r"Census Tract ([\d.]+)", col)
    return match.group(1) if match else col

def process_under5(df):
    cols_to_keep = [col for col in df.columns if '!!Estimate' in col and 'Margin of Error' not in col]
    filtered = df.iloc[[0, 1]][cols_to_keep].copy()
    no_commas = filtered.replace(',', '', regex=True)
    numeric = no_commas.apply(pd.to_numeric, errors='coerce')
    numeric.columns = [clean_header(c) for c in numeric.columns]
    cleaned = numeric.T
    cleaned.index.name = 'censustract'
    cleaned = cleaned.reset_index()
    cleaned = cleaned.rename(columns={0: 'total',1:'Owner_occupied'})
    return cleaned

homeownership_dfs = {2024: homeownership_2024, 2023: homeownership_2023, 2022: homeownership_2022, 2021: homeownership_2021, 2020: homeownership_2020}

homeownership_cleaned_list = []
for year, df in homeownership_dfs.items():
    cleaned = process_under5(df)
    cleaned = cleaned.rename(columns={
        'total': f'total_{year}',
        'Owner_occupied': f'Owner_occupied_{year}',
    })
    homeownership_cleaned_list.append(cleaned)

homeownership_long = homeownership_cleaned_list[0]
for df in homeownership_cleaned_list[1:]:
    homeownership_long = homeownership_long.merge(df, on='censustract', how='outer')

for year in [2024, 2023, 2022, 2021, 2020]:
    t, n1 = f'total_{year}', f'Owner_occupied_{year}'
    prop_owner_occupied = homeownership_long[n1] / homeownership_long[t]
    homeownership_long[f'prop_owner_occupied_{year}'] = round(prop_owner_occupied*100, 2)

#homeownership_long = homeownership_long[['censustract', 'prop_owner_occupied_2024', 'prop_owner_occupied_2023', 'prop_owner_occupied_2022', 'prop_owner_occupied_2021', 'prop_owner_occupied_2020']] 
homeownership_long  

,censustract,total_2024,Owner_occupied_2024,total_2023,Owner_occupied_2023,total_2022,Owner_occupied_2022,total_2021,Owner_occupied_2021,total_2020,Owner_occupied_2020,prop_owner_occupied_2024,prop_owner_occupied_2023,prop_owner_occupied_2022,prop_owner_occupied_2021,prop_owner_occupied_2020
0,9001,3196,2740,3001,2608,3234,2717,3467,2644,3616,2703,85.73,86.90,84.01,76.26,74.75
1,9002.01,3114,1279,3175,1762,2436,1293,2326,1383,2177,1387,41.07,55.50,53.08,59.46,63.71
2,9002.02,3668,2947,3520,2940,4124,3472,4271,3789,4254,3651,80.34,83.52,84.19,88.71,85.83
3,9002.03,5437,639,4886,366,4583,466,4574,536,4455,488,11.75,7.49,10.17,11.72,10.95
4,9003.01,3750,2341,3514,2305,3494,2110,3442,2111,3410,2014,62.43,65.59,60.39,61.33,59.06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
88,9017.02,4831,3069,4815,3158,4641,2864,4455,2449,4356,2307,63.53,65.59,61.71,54.97,52.96
89,9017.03,2750,2658,2574,2550,2418,2394,2770,2745,2518,2490,96.65,99.07,99.01,99.10,98.89
90,9017.04,7250,2533,7111,2963,6577,2787,7146,2834,6966,2386,34.94,41.67,42.37,39.66,34.25
91,9019,7248,3400,8132,3868,7720,3529,7148,3375,6694,3403,46.91,47.57,45.71,47.22,50.84


In [98]:
homeownership_long.to_csv('Housing/Homeownership Rate/Homeownership_rate_cleaned.csv', index=False)

#### Mobile Homes

In [95]:
mobile_2024 = pd.read_csv('Housing/Mobile Homes/ACSDT5Y2024.B25024.csv')
mobile_2023 = pd.read_csv('Housing/Mobile Homes/ACSDT5Y2023.B25024.csv')
mobile_2022 = pd.read_csv('Housing/Mobile Homes/ACSDT5Y2022.B25024.csv')
mobile_2021 = pd.read_csv('Housing/Mobile Homes/ACSDT5Y2021.B25024.csv')
mobile_2020 = pd.read_csv('Housing/Mobile Homes/ACSDT5Y2020.B25024.csv')
    

In [99]:
def clean_header(col):
    match = re.search(r"Census Tract ([\d.]+)", col)
    return match.group(1) if match else col

def process_under5(df):
    cols_to_keep = [col for col in df.columns if '!!Estimate' in col and 'Margin of Error' not in col]
    filtered = df.iloc[[0, 9]][cols_to_keep].copy()
    no_commas = filtered.replace(',', '', regex=True)
    numeric = no_commas.apply(pd.to_numeric, errors='coerce')
    numeric.columns = [clean_header(c) for c in numeric.columns]
    cleaned = numeric.T
    cleaned.index.name = 'censustract'
    cleaned = cleaned.reset_index()
    cleaned = cleaned.rename(columns={0: 'total',9:'Mobile_homes'})
    return cleaned

mobile_dfs = {2024: mobile_2024, 2023: mobile_2023, 2022: mobile_2022, 2021: mobile_2021, 2020: mobile_2020}

mobile_cleaned_list = []
for year, df in mobile_dfs.items():
    cleaned = process_under5(df)
    cleaned = cleaned.rename(columns={
        'total': f'total_{year}',
        'Mobile_homes': f'Mobile_homes_{year}',
    })
    mobile_cleaned_list.append(cleaned)

mobile_long = mobile_cleaned_list[0]
for df in mobile_cleaned_list[1:]:
    mobile_long = mobile_long.merge(df, on='censustract', how='outer')

for year in [2024, 2023, 2022, 2021, 2020]:
    t, n1 = f'total_{year}', f'Mobile_homes_{year}'
    prop_mobile_homes = mobile_long[n1] / mobile_long[t]
    mobile_long[f'prop_mobile_homes_{year}'] = round(prop_mobile_homes*100, 2)

#homeownership_long = homeownership_long[['censustract', 'prop_owner_occupied_2024', 'prop_owner_occupied_2023', 'prop_owner_occupied_2022', 'prop_owner_occupied_2021', 'prop_owner_occupied_2020']] 
mobile_long 

,censustract,total_2024,Mobile_homes_2024,total_2023,Mobile_homes_2023,total_2022,Mobile_homes_2022,total_2021,Mobile_homes_2021,total_2020,Mobile_homes_2020,prop_mobile_homes_2024,prop_mobile_homes_2023,prop_mobile_homes_2022,prop_mobile_homes_2021,prop_mobile_homes_2020
0,9001,1654,0,1623,0,1613,0,1580,0,1509,0,0.00,0.00,0.00,0.00,0.00
1,9002.01,1135,0,1180,0,1058,0,962,0,833,0,0.00,0.00,0.00,0.00,0.00
2,9002.02,1244,18,1229,21,1266,6,1269,0,1219,0,1.45,1.71,0.47,0.00,0.00
3,9002.03,1545,20,1540,13,1616,0,1679,0,1662,0,1.29,0.84,0.00,0.00,0.00
4,9003.01,1636,0,1619,0,1610,0,1621,0,1564,0,0.00,0.00,0.00,0.00,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
88,9017.02,1306,153,1302,190,1285,202,1310,174,1241,149,11.72,14.59,15.72,13.28,12.01
89,9017.03,744,0,754,0,711,0,697,0,707,0,0.00,0.00,0.00,0.00,0.00
90,9017.04,2069,44,2111,45,2126,0,2087,0,2047,0,2.13,2.13,0.00,0.00,0.00
91,9019,2434,11,2413,14,2378,10,2246,12,2213,0,0.45,0.58,0.42,0.53,0.00


In [100]:
mobile_long.to_csv('Housing/Mobile Homes/Mobile_homes_cleaned.csv', index=False)

#### Crowded Housing
$\text{Crowding rate} = \frac{\text{Occupied housing units with } >1.0 \text{ persons per room}}{\text{Occupied housing units}} \times 100$

$\text{Severely crowding rate} = \frac{\text{Occupied housing units with } >1.5 \text{ persons per room}}{\text{Occupied housing units}} \times 100$

In [101]:
crowded_2024 = pd.read_csv('Housing/Crowded Housing/ACSDT5Y2024.B25014.csv')
crowded_2023 = pd.read_csv('Housing/Crowded Housing/ACSDT5Y2023.B25014.csv')
crowded_2022 = pd.read_csv('Housing/Crowded Housing/ACSDT5Y2022.B25014.csv')
crowded_2021 = pd.read_csv('Housing/Crowded Housing/ACSDT5Y2021.B25014.csv')
crowded_2020 = pd.read_csv('Housing/Crowded Housing/ACSDT5Y2020.B25014.csv')

In [102]:
crowded_2024

,Label (Grouping),Census Tract 9001; Prince William County; Virginia!!Estimate,Census Tract 9001; Prince William County; Virginia!!Margin of Error,Census Tract 9002.01; Prince William County; Virginia!!Estimate,Census Tract 9002.01; Prince William County; Virginia!!Margin of Error,Census Tract 9002.02; Prince William County; Virginia!!Estimate,Census Tract 9002.02; Prince William County; Virginia!!Margin of Error,Census Tract 9002.03; Prince William County; Virginia!!Estimate,Census Tract 9002.03; Prince William County; Virginia!!Margin of Error,Census Tract 9003.01; Prince William County; Virginia!!Estimate,...,Census Tract 9017.02; Prince William County; Virginia!!Estimate,Census Tract 9017.02; Prince William County; Virginia!!Margin of Error,Census Tract 9017.03; Prince William County; Virginia!!Estimate,Census Tract 9017.03; Prince William County; Virginia!!Margin of Error,Census Tract 9017.04; Prince William County; Virginia!!Estimate,Census Tract 9017.04; Prince William County; Virginia!!Margin of Error,Census Tract 9019; Prince William County; Virginia!!Estimate,Census Tract 9019; Prince William County; Virginia!!Margin of Error,Census Tract 9801; Prince William County; Virginia!!Estimate,Census Tract 9801; Prince William County; Virginia!!Margin of Error
0,Total:,"1,627",±189,"1,051",±123,"1,075",±165,"1,490",±256,"1,599",...,"1,256",±197,740,±60,"1,942",±189,"2,385",±198,0,±13
1,Owner occupied:,"1,370",±214,397,±115,846,±167,150,±77,"1,024",...,787,±139,705,±73,747,±194,"1,173",±222,0,±13
2,0.50 or less occupants per room,"1,308",±208,263,±103,652,±162,67,±53,864,...,357,±77,437,±146,530,±181,721,±146,0,±13
3,0.51 to 1.00 occupants per room,62,±54,130,±75,156,±84,63,±52,160,...,315,±111,268,±154,184,±106,452,±182,0,±13
4,1.01 to 1.50 occupants per room,0,±13,4,±6,0,±13,20,±28,0,...,115,±103,0,±13,33,±47,0,±19,0,±13
5,1.51 to 2.00 occupants per room,0,±13,0,±13,32,±39,0,±19,0,...,0,±13,0,±13,0,±19,0,±19,0,±13
6,2.01 or more occupants per room,0,±13,0,±13,6,±12,0,±19,0,...,0,±13,0,±13,0,±19,0,±19,0,±13
7,Renter occupied:,257,±96,654,±128,229,±94,"1,340",±261,575,...,469,±213,35,±36,"1,195",±206,"1,212",±204,0,±13
8,0.50 or less occupants per room,249,±96,301,±86,171,±80,387,±136,323,...,210,±185,35,±36,361,±149,500,±160,0,±13
9,0.51 to 1.00 occupants per room,8,±14,240,±113,46,±40,780,±290,237,...,181,±115,0,±13,633,±204,578,±195,0,±13
